# S19 — background augmentation, paired comparison (H3)

This notebook answers the last open question in the project: **if you train the
model on leaves sitting in real field backgrounds instead of laboratory backdrops,
does the accuracy loss on field images get smaller?**

Everything else in the project is finished. H1 is confirmed (field accuracy is 78
points below laboratory accuracy), H2 is partly supported (removing the background
costs 56.5 points in the lab), H4 is confirmed (a confidence threshold does not
rescue a usable subset). H3 is the only hypothesis with no result, because it needs
a training experiment rather than an evaluation.

## What it produces

Six trained models and one comparison table.

| arm | what the model trains on | everything else |
|---|---|---|
| **baseline** | the same PlantVillage images as always | protocol v2, untouched |
| **composite** | the same leaves, pasted onto random field backgrounds | protocol v2, untouched |

Three paired seeds each. Same architecture, same learning rate, same epochs, same
evaluator, same 236 field images. The only thing that differs is what is behind the
leaf.

The comparison is **paired**: for every field image, both models either get it
right or wrong, and the test resamples the same images for both. That stops a seed
that happened to see easy field images from flattering the treatment.

## What you need before pressing Run All

| requirement | how to set it |
|---|---|
| Accelerator | **GPU T4 x2** (Settings, right panel) |
| Dataset 1 | the PlantVillage dataset already used by notebooks 06 to 08 |
| Dataset 2 | **DeepWeeds**, Kaggle slug `imsparsh/deepweeds` |
| Internet | not needed |
| Run mode | **Save Version, then Save and Run All** so it survives the browser closing |

If DeepWeeds is not attached, cell 1 stops with a message and nothing else runs.
That is deliberate: without a licence-clean background corpus there is no
experiment, and training on PlantDoc imagery instead would corrupt the field
evaluation that produced the headline result.

## How long it takes

About 2.5 hours for six runs. If you want a fast smoke version first, set
`SEEDS = [42]` in cell 1: one pair of runs, roughly 50 minutes, enough to see
whether the direction of the effect is even interesting.

The notebook prints progress as it goes. You can close the browser once the commit
starts.

## What can go wrong, and what the notebook does about it

| failure | handling |
|---|---|
| DeepWeeds missing | cell 1 stops immediately, before any time is spent |
| a background image came from PlantDoc | the hash check in cell 2 aborts before training |
| the recipe cannot learn | the gate in cell 5 stops it in about 5 minutes |
| one arm crashes | the other arm still runs, so partial results survive |
| the session dies | finished seeds are skipped on restart |
| evaluation fails for one seed | that pair is skipped, the rest still compare |
| everything fails | the final cell still writes a bundle with whatever exists |

## Honest limits of the method

These belong in the report next to the result, not in a footnote.

**One random background per image, fixed for the run.** Re-sampling the background
every epoch would be stronger augmentation, but it needs the training engine to
carry a background pool and masks through the loop. With a fixed random assignment
the background-to-class mapping is already destroyed, because every class receives
random backdrops. What is given up is per-image background variety.

**The leaf mask is a heuristic.** Corner-colour plus greenness fails on close-up
leaf textures where the frame is almost entirely leaf. Masks covering less than 25%
or more than 90% of the frame are not trusted, and those images pass through
unaltered rather than being composited wrongly. On the validation split that was
2.7% of images.

**DeepWeeds is Australian rangeland.** Different continent, different species,
different cameras from both PlantVillage and PlantDoc. That is what makes it safe
to use, and it also means the backgrounds are not the same kind of field as
PlantDoc's, so a positive result would be a lower bound on what real in-domain
backgrounds could do.

## Before you press Run All

Work through this list once. Every item on it has cost this project time before.

1. **Accelerator set to GPU T4 x2** in the right-hand Settings panel. Cell 1 checks
   this and stops if only one GPU is present.
2. **PlantVillage attached.** Cell 1 checks for the development manifest and finds
   the dataset root automatically if the path has changed.
3. **DeepWeeds attached** at `/kaggle/input/deepweeds`. Cell 1 stops if it is
   missing, and tells you the slug to add.
4. **Run All, not one cell at a time.** Cells 3 to 7 depend on values defined
   earlier; the notebook is written for a top-to-bottom run.
5. **Save Version, then Save and Run All** if you want it to continue after you
   close the tab. An interactive session dies with the browser.

You can safely re-run this notebook. Finished seeds are skipped, the caches are
reused, and the composited images are not rebuilt if they already exist.

## 1. Setup and preflight

Sets the seeds and epoch count, finds the dataset, and checks the three things that make the run possible: two GPUs, the PlantVillage manifest, and the background corpus.

**Expected output:** the GPU names, the image counts, and the estimated runtime.

**If it stops here:** read the error. It names the exact path or setting that is missing.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torchvision

SEEDS = [42, 43, 44]          # reduce to [42] for a one-pair smoke run
EPOCHS = 12
GATE_ACCURACY = 0.95

DATASET_ROOT = Path("/kaggle/input/datasets/shubhamkumar12864/plantvillage")
KAGGLE_INPUT = Path("/kaggle/input")
BACKGROUND_ROOT = Path("/kaggle/input/deepweeds")      # attach DeepWeeds here


def autodiscover_dataset_root(preferred: Path) -> Path:
    if (preferred / "data" / "manifests" / "plantvillage_development.csv").is_file():
        return preferred
    if KAGGLE_INPUT.is_dir():
        for manifests in sorted(KAGGLE_INPUT.rglob("manifests")):
            if (manifests / "plantvillage_development.csv").is_file():
                return manifests.parent.parent
    return preferred


DATASET_ROOT = autodiscover_dataset_root(DATASET_ROOT)
MANIFEST_DIR = DATASET_ROOT / "data" / "manifests"
IMAGE_ROOT = DATASET_ROOT / "data" / "raw" / "PlantVillage"
TRAIN_MANIFEST = MANIFEST_DIR / "plantvillage_development.csv"

WORKDIR = Path("/kaggle/working")
SCRIPT = WORKDIR / "train_ddp.py"
WORKER = WORKDIR / "sweep_worker.py"
PROTOCOL_PATH = WORKDIR / "protocol_v2.json"
MASK_SCRIPT = WORKDIR / "make_leaf_masks.py"
COMPOSITE_SCRIPT = WORKDIR / "build_composite_cache.py"
DISJOINT_SCRIPT = WORKDIR / "check_background_disjointness.py"
EVAL_SCRIPT = WORKDIR / "evaluate_plantdoc.py"
BOOTSTRAP_SCRIPT = WORKDIR / "paired_bootstrap.py"

CACHE_DIR = Path("/kaggle/temp/plantvillage_cache")
DERIVED = Path("/kaggle/temp/derived")
OUTPUT_ROOT = WORKDIR / "runs_bg"
IMAGE_SIZE = 224
STARTED_AT = time.time()

for folder in (CACHE_DIR, DERIVED, OUTPUT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Set Accelerator -> GPU T4 x2 in Kaggle Settings, then Run All again.")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(2)])

if not TRAIN_MANIFEST.is_file():
    raise FileNotFoundError(f"PlantVillage not attached: expected {TRAIN_MANIFEST}")
if not BACKGROUND_ROOT.is_dir():
    raise FileNotFoundError(
        f"Background corpus not attached at {BACKGROUND_ROOT}.\n"
        "S19 cannot run without a licence-clean corpus disjoint from PlantDoc. "
        "See docs/BACKGROUND_CORPUS_OPTIONS.md."
    )

dev = pd.read_csv(TRAIN_MANIFEST)
train_df = dev.loc[dev.split == "train"].reset_index(drop=True)
val_df = dev.loc[dev.split == "validation"].reset_index(drop=True)
classes = sorted(train_df.label.unique().tolist())
if len(classes) != 38:
    raise ValueError(f"expected 38 classes, found {len(classes)}")
print(f"PlantVillage: {len(train_df):,} train / {len(val_df):,} validation")
print(f"background corpus: {BACKGROUND_ROOT}")
print(f"seeds: {SEEDS}  |  epochs: {EPOCHS}  |  estimated "
      f"{len(SEEDS) * 2 * EPOCHS * 2.2 / 60:.1f} h")

## 2. Disjointness gate

Hashes every background image and compares it against the 53,999 hashes already recorded in the PlantDoc and PlantVillage manifests.

This is the check that makes H3 legitimate. If one background came from PlantDoc, the field evaluation would be seeing training data and the 78-point gap would be void. "They are different datasets" is not evidence; a hash comparison is.

**Expected output:** `PASS: no byte-identical overlap`, then the limitation note that exact hashing will not catch a re-encoded near-duplicate.

In [ ]:
print("=" * 68)
print("Disjointness gate: if a background came from PlantDoc, the whole")
print("comparison is void, so this runs before anything trains.")
print("=" * 68)

disjoint_out = DERIVED / "disjointness.json"
check = subprocess.run([sys.executable, str(DISJOINT_SCRIPT),
                        "--backgrounds", str(BACKGROUND_ROOT),
                        "--out", str(disjoint_out)],
                       env={**os.environ, "PYTHONUNBUFFERED": "1"})
if check.returncode != 0:
    raise RuntimeError(
        "DEDISJOINTNESS FAILED: a background image is byte-identical to a PlantDoc or "
        "PlantVillage image. Training on it would put the evaluation set into the training "
        "data. See " + str(disjoint_out)
)
DISJOINT = json.loads(disjoint_out.read_text())
print(f"\nchecked {DISJOINT['candidate_images']:,} backgrounds against "
      f"{DISJOINT['plantdoc_hashes']:,} PlantDoc and {DISJOINT['plantvillage_hashes']:,} "
      f"PlantVillage hashes: {DISJOINT['verdict']}")
print("limitation:", DISJOINT["limitation"])

## 3a. Clean caches

Decodes every image once into a uint8 memory map. The validation cache is built here and is **never** composited, so the PlantVillage score stays comparable to the baseline.

**Expected output:** `clean caches ready`. Takes about 5 minutes the first time and is reused afterwards.

In [ ]:
def cache_split(frame, image_root, tag):
    labels = np.asarray([classes.index(label) for label in frame.label], dtype=np.int64)
    paths = [image_root / rel for rel in frame.relative_path]
    path_key = hashlib.sha256("\n".join(map(str, paths)).encode()).hexdigest()[:16]
    stem = CACHE_DIR / f"{tag}_{IMAGE_SIZE}_{len(paths)}_{path_key}"
    x_path, y_path, done = Path(f"{stem}_x.npy"), Path(f"{stem}_y.npy"), Path(f"{stem}.done")
    shape = (len(paths), IMAGE_SIZE, IMAGE_SIZE, 3)

    if done.is_file() and x_path.is_file() and y_path.is_file():
        existing = np.load(x_path, mmap_mode="r", allow_pickle=False)
        if existing.shape == shape and np.array_equal(np.load(y_path, allow_pickle=False), labels):
            print(f"reused {tag} cache")
            return x_path, y_path
        del existing

    array = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.uint8, shape=shape)
    failures = []

    def decode(item):
        i, path = item
        try:
            with Image.open(path) as image:
                array[i] = np.asarray(image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE),
                                                                  Image.Resampling.BILINEAR))
        except Exception as exc:
            failures.append((str(path), repr(exc)))
        return i

    with ThreadPoolExecutor(max_workers=8) as pool:
        list(tqdm(pool.map(decode, enumerate(paths)), total=len(paths), desc=tag, unit="img"))
    array.flush(); del array
    if failures:
        raise RuntimeError(f"{len(failures)} images failed to decode, first: {failures[0]}")
    np.save(y_path, labels, allow_pickle=False)
    done.write_text("complete")
    return x_path, y_path


TRAIN_X_PATH, TRAIN_Y_PATH = cache_split(train_df, IMAGE_ROOT / "train", "train")
VAL_X_PATH, VAL_Y_PATH = cache_split(val_df, IMAGE_ROOT / "train", "val")
CLASSES_PATH = CACHE_DIR / "classes.json"
CLASSES_PATH.write_text(json.dumps(classes, indent=2))
print("clean caches ready")

## 3b. Leaf masks and the composite cache

Two steps. First, segment the leaf from the backdrop in every training image using corner colour plus greenness. Second, paste each leaf onto a random field background.

Images whose mask covers less than 25% or more than 90% of the frame are not trusted and pass through unchanged, because that is where the heuristic fails.

**Expected output:** how many images were composited and how many passed through, plus two check images in `/kaggle/temp/derived/` you can open to confirm the leaves survive and the backgrounds actually changed.

In [ ]:
MASKS_PATH = DERIVED / "masks_train.npz"
COMPOSITE_X = CACHE_DIR / "train_composite_x.npy"
COMPOSITE_Y = TRAIN_Y_PATH

if not MASKS_PATH.is_file():
    print("building leaf masks for the training split (~4 minutes)")
    subprocess.run([sys.executable, str(MASK_SCRIPT), "--split", "train",
                    "--manifest", str(TRAIN_MANIFEST), "--image-root", str(IMAGE_ROOT / "train"),
                    "--out", str(MASKS_PATH),
                    "--check-sheet", str(DERIVED / "mask_check.png")], check=True)

if not COMPOSITE_X.is_file():
    print("compositing leaves onto field backgrounds")
    subprocess.run([sys.executable, str(COMPOSITE_SCRIPT),
                    "--backgrounds", str(BACKGROUND_ROOT), "--masks", str(MASKS_PATH),
                    "--train-cache", str(TRAIN_X_PATH), "--out", str(COMPOSITE_X),
                    "--check-sheet", str(DERIVED / "composite_check.png")], check=True)

COMPOSITE_SUMMARY = json.loads(Path(str(COMPOSITE_X) + ".json").read_text())
print(f"composited {COMPOSITE_SUMMARY['images_composited']:,} images, "
      f"{COMPOSITE_SUMMARY['images_passed_through']:,} passed through "
      f"(mask outside {COMPOSITE_SUMMARY['mask_trust_band']})")

## 4. Scripts and protocol

Writes the eight scripts this notebook runs, compiles each one, and asserts that protocol v2 is unchanged. If this cell does not print `all scripts compile`, stop.

In [ ]:
# Writes every script this notebook uses, then checks them.
FILES = {
    '/kaggle/working/train_ddp.py': '#!/usr/bin/env python3\n"""Fine-tune a supported CNN on PlantVillage with two-GPU DDP.\n\nThis is the v2 training engine. It replaces the pilot\n``train_efficientnet_b0_ddp.py`` and is architecture-agnostic, so the\nEfficientNet-B0 run and the ResNet18 run share one auditable code path.\n\nWhy v2 exists (measured problems in the pilot run)\n--------------------------------------------------\n* The pilot used ``num_workers=0`` on a 5.5 GB memory-mapped cache. Every batch\n  page-faulted in the main process, so the two T4s idled while data loaded.\n  Per-epoch time grew 1.7 -> 15.65 min as the page cache evicted. v2 uses\n  worker processes with prefetch, which is the standard fix and should land\n  around 1 min/epoch.\n* The pilot set ``cudnn.benchmark = False`` and used a global batch of 32.\n  v2 enables benchmark (input shape is fixed) and defaults to 64 per GPU.\n* The loss was plain, unweighted cross-entropy on a 38.4x imbalanced training\n  set (Orange HLB 3,842 vs Potato healthy 100). v2 supports class-balanced\n  cross-entropy so macro-F1, the model-selection metric, is not starved by\n  rare classes.\n* The pilot saved only headline metrics. v2 also writes per-class\n  precision/recall/F1/support, the confusion matrix and calibration\n  diagnostics, which the report and the guide review need anyway.\n\nWhat v2 adds for accuracy\n-------------------------\nClass-balanced cross-entropy, label smoothing, cosine schedule with linear\nwarmup, gradient clipping, and an EMA copy of the weights that is evaluated and\nkept when it beats the raw weights.\n\nWhat v2 adds for safety and reproducibility\n-------------------------------------------\n``weights_only=True`` checkpoint loading with an explicit fallback, and a\ndataset fingerprint stored in the checkpoint so a resumed run cannot silently\ncontinue against a different split.\n\nUsage\n-----\n    torchrun --standalone --nproc_per_node=2 scripts/train_ddp.py \\\n        --arch efficientnet_b0 \\\n        --train-images /path/train_x.npy --train-labels /path/train_y.npy \\\n        --val-images /path/val_x.npy --val-labels /path/val_y.npy \\\n        --classes-json /path/classes.json --output-dir /path/out\n\nAdd ``--dry-run`` to execute three optimizer steps and exit, which is the\ncheap way to prove a Kaggle session is wired up before committing GPU hours.\n"""\nfrom __future__ import annotations\n\nimport os\n\n# Keep BLAS single-threaded per rank: DDP already saturates the cores and\n# oversubscription is a common cause of stalled steps.\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\n\nimport argparse\nimport contextlib\nimport copy\nimport hashlib\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn.functional as F\nfrom sklearn.metrics import (\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_fscore_support,\n    top_k_accuracy_score,\n)\nfrom torch import nn\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, Subset\nfrom tqdm.auto import tqdm\n\ntry:  # torchvision is a hard dependency in practice; import lazily for CLI help\n    import torchvision\n    from torchvision import models\nexcept Exception as exc:  # pragma: no cover - only hit in broken installs\n    torchvision = None\n    models = None\n    _TV_IMPORT_ERROR = exc\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\nSMALLCNN_MEAN = (0.5, 0.5, 0.5)\nSMALLCNN_STD = (0.5, 0.5, 0.5)\n\nARCHES = ("smallcnn", "resnet18", "efficientnet_b0", "convnext_tiny")\n\n\n# --------------------------------------------------------------------------- #\n# Data\n# --------------------------------------------------------------------------- #\nclass CachedImages(Dataset):\n    """Read one uint8 image at a time from a disk-backed NPY memory map."""\n\n    def __init__(self, image_file, labels_file, image_size: int):\n        self.images = np.load(image_file, mmap_mode="r", allow_pickle=False)\n        self.labels = np.load(labels_file, mmap_mode="r", allow_pickle=False)\n        expected = (image_size, image_size, 3)\n        if self.images.ndim != 4 or tuple(self.images.shape[1:]) != expected:\n            raise ValueError(f"Bad cached image shape: {self.images.shape}, expected (*, {expected})")\n        if self.images.dtype != np.uint8:\n            raise ValueError(f"Image cache must be uint8, found {self.images.dtype}")\n        if len(self.images) != len(self.labels):\n            raise ValueError("Image cache and label cache have different lengths")\n\n    def __len__(self) -> int:\n        return len(self.labels)\n\n    def __getitem__(self, index: int):\n        # copy() detaches the row from the memory map so the loader can batch it\n        image = torch.from_numpy(self.images[index].copy())\n        return image, int(self.labels[index])\n\n\nclass SmallCNN(nn.Module):\n    """Four-stage CNN baseline.\n\n    Defined here as well as in src/plant_disease/models/smallcnn.py because this\n    file has to run standalone inside Kaggle, where the package is not installed.\n    scripts/smoke_test_engine.py asserts the two definitions are parameter-\n    identical, so they cannot drift apart silently.\n    """\n\n    def __init__(self, num_classes: int):\n        super().__init__()\n        blocks = []\n        channels = 3\n        for output_channels in (32, 64, 128, 256):\n            blocks.extend([\n                nn.Conv2d(channels, output_channels, 3, padding=1, bias=False),\n                nn.BatchNorm2d(output_channels),\n                nn.ReLU(inplace=True),\n                nn.MaxPool2d(2),\n            ])\n            channels = output_channels\n        self.features = nn.Sequential(*blocks)\n        self.head = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.30), nn.Linear(256, num_classes)\n        )\n\n    def forward(self, images):\n        return self.head(self.features(images))\n\n\ndef build_model(arch: str, num_classes: int, pretrained: bool) -> nn.Module:\n    if arch == "smallcnn":\n        # Random initialisation by design; there is no pretrained SmallCNN.\n        return SmallCNN(num_classes)\n    if models is None:  # pragma: no cover\n        raise RuntimeError(f"torchvision import failed: {_TV_IMPORT_ERROR}")\n    if arch == "efficientnet_b0":\n        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None\n        model = models.efficientnet_b0(weights=weights)\n        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)\n    elif arch == "resnet18":\n        weights = models.ResNet18_Weights.DEFAULT if pretrained else None\n        model = models.resnet18(weights=weights)\n        model.fc = nn.Linear(model.fc.in_features, num_classes)\n    elif arch == "convnext_tiny":\n        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None\n        model = models.convnext_tiny(weights=weights)\n        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)\n    else:\n        raise ValueError(f"Unsupported arch {arch!r}; choose from {ARCHES}")\n    return model\n\n\ndef dataset_fingerprint(train_labels, val_labels, classes, image_size: int) -> str:\n    """Stable identifier for the exact split a checkpoint was trained on."""\n    digest = hashlib.sha256()\n    digest.update(np.ascontiguousarray(train_labels).tobytes())\n    digest.update(np.ascontiguousarray(val_labels).tobytes())\n    digest.update(json.dumps(classes, separators=(",", ":")).encode("utf-8"))\n    digest.update(str(image_size).encode("utf-8"))\n    return digest.hexdigest()\n\n\ndef block_shuffled_indices(n: int, rng: np.random.Generator, block: int = 2048) -> np.ndarray:\n    """Permute block order, then shuffle inside each block.\n\n    Reading in blocks keeps the page cache local instead of jumping across the\n    whole 5 GB map every batch, which was the pilot\'s failure mode.\n    """\n    order = np.arange(n)\n    starts = np.arange(0, n, block)\n    rng.shuffle(starts)\n    out = []\n    for start in starts:\n        chunk = order[start:start + block]\n        rng.shuffle(chunk)\n        out.append(chunk)\n    return np.concatenate(out)\n\n\n# --------------------------------------------------------------------------- #\n# Augmentation and normalisation (all on GPU)\n# --------------------------------------------------------------------------- #\ndef amp_context(use_cuda: bool):\n    """FP16 autocast on the accelerator, no-op on CPU (CPU autocast is slower)."""\n    if use_cuda:\n        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)\n    return contextlib.nullcontext()\n\n\ndef gpu_augment_and_normalize(image_u8, device, training: bool, mean, std, rotation_deg: float = 12.0):\n    """NHWC uint8 -> normalised NCHW float, with augmentation applied on the GPU.\n\n    Augmentation runs on the accelerator so no CPU worker is the bottleneck and\n    the worker processes only have to memcpy bytes out of the memory map.\n    """\n    x = image_u8.to(device=device, dtype=torch.float32, non_blocking=True)\n    x = x.permute(0, 3, 1, 2).div_(255.0).contiguous(memory_format=torch.channels_last)\n    if training:\n        batch = x.shape[0]\n        # horizontal flip\n        flip = torch.rand(batch, device=device) < 0.5\n        x = torch.where(flip.view(batch, 1, 1, 1), x.flip(3), x)\n        # small rotation\n        if rotation_deg > 0:\n            angle = (torch.rand(batch, device=device) * 2.0 - 1.0) * math.radians(rotation_deg)\n            cosine, sine = angle.cos(), angle.sin()\n            theta = torch.zeros(batch, 2, 3, device=device)\n            theta[:, 0, 0], theta[:, 0, 1] = cosine, -sine\n            theta[:, 1, 0], theta[:, 1, 1] = sine, cosine\n            grid = F.affine_grid(theta, x.shape, align_corners=False)\n            x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n        # brightness and per-channel contrast jitter\n        brightness = torch.rand(batch, 1, 1, 1, device=device) * 0.24 + 0.88\n        x = (x * brightness).clamp_(0.0, 1.0)\n        gain = torch.rand(batch, 3, 1, 1, device=device) * 0.16 + 0.92\n        x = (x * gain).clamp_(0.0, 1.0)\n    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)\n    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)\n    return ((x - mean_t) / std_t).contiguous(memory_format=torch.channels_last)\n\n\n# --------------------------------------------------------------------------- #\n# Metrics and calibration\n# --------------------------------------------------------------------------- #\ndef expected_calibration_error(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:\n    """Binned |accuracy - confidence|, weighted by bin population."""\n    confidence = probabilities.max(axis=1)\n    predictions = probabilities.argmax(axis=1)\n    correct = (predictions == y_true).astype(np.float64)\n    edges = np.linspace(0.0, 1.0, n_bins + 1)\n    ece = 0.0\n    for low, high in zip(edges[:-1], edges[1:]):\n        mask = (confidence > low) & (confidence <= high)\n        if mask.any():\n            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())\n    return float(ece)\n\n\ndef brier_score(probabilities: np.ndarray, y_true: np.ndarray) -> float:\n    """Mean squared error between predicted probabilities and one-hot labels."""\n    one_hot = np.zeros_like(probabilities)\n    one_hot[np.arange(len(y_true)), y_true] = 1.0\n    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))\n\n\ndef score_predictions(y_true, y_pred, probabilities=None) -> dict:\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    metrics = {\n        "accuracy": float((y_true == y_pred).mean()),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_precision": float(precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)[0]),\n        "macro_recall": float(precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)[1]),\n        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),\n    }\n    if probabilities is not None:\n        scores = probabilities.astype(np.float64)\n        labels = np.arange(scores.shape[1])\n        metrics["top3_accuracy"] = float(top_k_accuracy_score(y_true, scores, k=3, labels=labels))\n        metrics["ece"] = expected_calibration_error(scores, y_true)\n        metrics["brier"] = brier_score(scores, y_true)\n        metrics["mean_confidence"] = float(scores.max(axis=1).mean())\n        metrics["frac_below_0_5"] = float((scores.max(axis=1) < 0.5).mean())\n    return metrics\n\n\ndef classification_table(y_true, y_pred, classes) -> pd.DataFrame:\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y_true, y_pred, labels=np.arange(len(classes)), zero_division=0\n    )\n    return pd.DataFrame(\n        {\n            "class": classes,\n            "precision": precision,\n            "recall": recall,\n            "f1": f1,\n            "support": support,\n        }\n    )\n\n\n# --------------------------------------------------------------------------- #\n# EMA\n# --------------------------------------------------------------------------- #\nclass ModelEma:\n    """Exponential moving average of model weights, kept on the evaluating rank."""\n\n    def __init__(self, model: nn.Module, decay: float = 0.999):\n        self.module = copy.deepcopy(model).eval()\n        for parameter in self.module.parameters():\n            parameter.requires_grad_(False)\n        self.decay = decay\n        self.updates = 0\n\n    @torch.no_grad()\n    def update(self, model: nn.Module):\n        self.updates += 1\n        # Ramp the effective decay early so the EMA is not anchored to the\n        # random initialisation for the first few hundred steps.\n        decay = min(self.decay, (1 + self.updates) / (10 + self.updates))\n        for ema_parameter, parameter in zip(self.module.parameters(), model.parameters()):\n            ema_parameter.mul_(decay).add_(parameter.detach(), alpha=1.0 - decay)\n        for ema_buffer, buffer in zip(self.module.buffers(), model.buffers()):\n            ema_buffer.copy_(buffer)\n\n\n# --------------------------------------------------------------------------- #\n# Evaluation\n# --------------------------------------------------------------------------- #\n@torch.no_grad()\ndef evaluate(model, dataset, device, mean, std, batch_size, workers, use_cuda=True) -> tuple:\n    model.eval()\n    loader = DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=workers,\n        pin_memory=use_cuda,\n        persistent_workers=workers > 0,\n    )\n    y_true, y_pred, probability_rows = [], [], []\n    started = time.perf_counter()\n    progress = tqdm(loader, desc="validation", unit="batch", leave=True, mininterval=1.0, disable=None)\n    for images, labels in progress:\n        x = gpu_augment_and_normalize(images, device, training=False, mean=mean, std=std)\n        with amp_context(use_cuda):\n            logits = model(x)\n        probabilities = torch.softmax(logits.float(), dim=1).cpu().numpy()\n        y_true.extend(labels.numpy().tolist())\n        y_pred.extend(probabilities.argmax(axis=1).tolist())\n        probability_rows.append(probabilities)\n        seen = len(y_true)\n        progress.set_postfix(\n            val_accuracy=f"{np.mean(np.asarray(y_true) == np.asarray(y_pred)):.4f}",\n            img_s=f"{seen / max(time.perf_counter() - started, 1e-6):.1f}",\n            refresh=False,\n        )\n    probabilities = np.concatenate(probability_rows, axis=0)\n    return score_predictions(y_true, y_pred, probabilities), np.asarray(y_true), np.asarray(y_pred), probabilities\n\n\n# --------------------------------------------------------------------------- #\n# Checkpoint IO\n# --------------------------------------------------------------------------- #\ndef atomic_save(payload, path: Path):\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\ndef safe_load(path: Path, map_location):\n    """Load a checkpoint, preferring the non-executing ``weights_only`` path."""\n    try:\n        return torch.load(path, map_location=map_location, weights_only=True)\n    except Exception as exc:  # older torch or an exotic payload\n        print(f"  weights_only=True failed for {path.name} ({exc}); retrying in compatibility mode", flush=True)\n        return torch.load(path, map_location=map_location, weights_only=False)\n\n\ndef clean_state_dict(state):\n    if state and all(str(key).startswith("module.") for key in state):\n        return {key[7:]: value for key, value in state.items()}\n    return state\n\n\n# --------------------------------------------------------------------------- #\n# Main\n# --------------------------------------------------------------------------- #\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--train-images", required=True)\n    parser.add_argument("--train-labels", required=True)\n    parser.add_argument("--val-images", required=True)\n    parser.add_argument("--val-labels", required=True)\n    parser.add_argument("--classes-json", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--arch", default="efficientnet_b0", choices=ARCHES)\n    parser.add_argument("--epochs", type=int, default=12)\n    parser.add_argument("--patience", type=int, default=3)\n    parser.add_argument("--local-batch-size", type=int, default=64, help="per-GPU batch; global = this x world size")\n    parser.add_argument("--eval-batch-size", type=int, default=128)\n    parser.add_argument("--lr", type=float, default=3e-4)\n    parser.add_argument("--min-lr-factor", type=float, default=0.01)\n    parser.add_argument("--weight-decay", type=float, default=0.05)\n    parser.add_argument("--warmup-fraction", type=float, default=0.10)\n    parser.add_argument("--label-smoothing", type=float, default=0.05)\n    parser.add_argument("--grad-clip", type=float, default=5.0)\n    parser.add_argument("--class-weight-alpha", type=float, default=0.5,\n                        help="0 disables class-balanced loss; 1 is full inverse frequency")\n    parser.add_argument("--ema-decay", type=float, default=0.999, help="0 disables EMA")\n    parser.add_argument("--rotation-deg", type=float, default=12.0)\n    parser.add_argument("--workers", type=int, default=4, help="DataLoader workers per rank")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--compile", action="store_true", help="torch.compile the model (slower first step)")\n    parser.add_argument("--shuffle-mode", default="blocked", choices=("random", "blocked"))\n    parser.add_argument("--dry-run", action="store_true", help="run 3 optimizer steps then exit")\n    parser.add_argument("--device", default="auto", choices=("auto", "cuda", "cpu"),\n                        help="auto uses CUDA when present; cpu exists for local smoke tests")\n    # Escape hatch for environments where torchrun\'s TCP store is unavailable\n    # (some Windows CPU wheels are built without libuv). A file-based store keeps\n    # the same code path reachable without the launcher.\n    parser.add_argument("--init-method", default="env://",\n                        help="URL for process-group init; non-env values require --rank/--world-size")\n    parser.add_argument("--rank", type=int, default=None)\n    parser.add_argument("--world-size", type=int, default=None)\n    parser.add_argument("--no-dist", action="store_true",\n                        help="single process, no process group; for local code checks only")\n    args = parser.parse_args()\n\n    use_cuda = args.device == "cuda" or (args.device == "auto" and torch.cuda.is_available())\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. In Kaggle select Accelerator -> GPU T4 x2.")\n    if not use_cuda and args.device == "auto":\n        raise RuntimeError(\n            "No CUDA device. Pass --device cpu to run the engine on CPU (slow, intended for a "\n            "small local check) or select GPU T4 x2 in Kaggle."\n        )\n\n    backend = "nccl" if use_cuda else "gloo"\n    distributed = not args.no_dist\n    if not distributed:\n        rank, world, local_rank = 0, 1, 0\n    elif args.init_method == "env://":\n        dist.init_process_group(backend=backend)\n        rank = dist.get_rank()\n        world = dist.get_world_size()\n        local_rank = int(os.environ.get("LOCAL_RANK", 0))\n    else:\n        if args.rank is None or args.world_size is None:\n            raise ValueError("--init-method other than env:// requires --rank and --world-size")\n        dist.init_process_group(\n            backend=backend, init_method=args.init_method, rank=args.rank, world_size=args.world_size\n        )\n        rank, world, local_rank = args.rank, args.world_size, args.rank\n\n    def synchronise():\n        """Barrier when a process group exists, otherwise a no-op."""\n        if distributed:\n            dist.barrier()\n\n    def unwrap(module: nn.Module) -> nn.Module:\n        return module.module if isinstance(module, DDP) else module\n    if use_cuda:\n        torch.cuda.set_device(local_rank)\n        device = torch.device("cuda", local_rank)\n    else:\n        device = torch.device("cpu")\n    torch.set_num_threads(1)\n    random.seed(args.seed + rank)\n    np.random.seed(args.seed + rank)\n    torch.manual_seed(args.seed + rank)\n    if use_cuda:\n        torch.cuda.manual_seed_all(args.seed + rank)\n        # Fixed 224x224 input means cuDNN can pick and keep the fastest algorithm.\n        torch.backends.cudnn.benchmark = True\n        torch.backends.cuda.matmul.allow_tf32 = True\n    torch.set_float32_matmul_precision("high")\n\n    classes = json.loads(Path(args.classes_json).read_text(encoding="utf-8"))\n    train_ds = CachedImages(args.train_images, args.train_labels, args.image_size)\n    val_ds = CachedImages(args.val_images, args.val_labels, args.image_size)\n    if len(train_ds) == 0 or len(val_ds) == 0:\n        raise ValueError("Both train and validation caches must contain images")\n    if int(np.asarray(train_ds.labels).max()) >= len(classes) or int(np.asarray(val_ds.labels).max()) >= len(classes):\n        raise ValueError("A cached label index is out of range for the class list")\n\n    fingerprint = dataset_fingerprint(\n        np.asarray(train_ds.labels), np.asarray(val_ds.labels), classes, args.image_size\n    )\n\n    mean, std = (SMALLCNN_MEAN, SMALLCNN_STD) if args.arch == "smallcnn" else (IMAGENET_MEAN, IMAGENET_STD)\n\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    best_path = output_dir / f"{args.arch}_best.pt"\n    last_path = output_dir / f"{args.arch}_last.pt"\n\n    resume_config = {\n        "architecture": args.arch,\n        "classes": classes,\n        "seed": args.seed,\n        "image_size": args.image_size,\n        "local_batch_size": args.local_batch_size,\n        "global_batch_size": args.local_batch_size * world,\n        "optimizer": "AdamW",\n        "learning_rate": args.lr,\n        "weight_decay": args.weight_decay,\n        "label_smoothing": args.label_smoothing,\n        "class_weight_alpha": args.class_weight_alpha,\n        "ema_decay": args.ema_decay,\n        "max_epochs": args.epochs,\n        "patience": args.patience,\n        "dataset_fingerprint": fingerprint,\n        "torch_version": str(torch.__version__),\n        "torchvision_version": str(torchvision.__version__),\n    }\n\n    resume = None\n    if last_path.is_file() and not args.dry_run:\n        resume = safe_load(last_path, "cpu")\n        stored = resume.get("resume_config", {})\n        if stored.get("dataset_fingerprint") not in (None, fingerprint):\n            raise RuntimeError(\n                "The resume checkpoint was trained on a different split "\n                f"({stored.get(\'dataset_fingerprint\')} != {fingerprint}). "\n                "Move the old run aside or use a fresh --output-dir."\n            )\n        if {k: v for k, v in stored.items() if k != "dataset_fingerprint"} != {\n            k: v for k, v in resume_config.items() if k != "dataset_fingerprint"\n        }:\n            raise RuntimeError("Resume checkpoint settings differ from this run. Use a fresh --output-dir.")\n        if rank == 0:\n            print(f"Resuming from completed epoch {resume[\'epoch\']}.", flush=True)\n\n    # Class-balanced loss, softened by --class-weight-alpha. alpha=0 is plain CE.\n    train_labels = np.asarray(train_ds.labels)\n    class_counts = np.bincount(train_labels, minlength=len(classes)).astype(np.float64)\n    if args.class_weight_alpha > 0:\n        present = class_counts > 0\n        weights = np.ones(len(classes), dtype=np.float64)\n        weights[present] = (class_counts.sum() / (len(classes) * class_counts[present])) ** args.class_weight_alpha\n        weights = weights / weights[present].mean()\n    else:\n        weights = np.ones(len(classes), dtype=np.float64)\n    loss_weight = torch.tensor(weights, dtype=torch.float32, device=device)\n\n    pretrained = rank == 0 and resume is None\n    base = build_model(args.arch, len(classes), pretrained=pretrained)\n    base = base.to(device=device, memory_format=torch.channels_last)\n    if resume is not None:\n        base.load_state_dict(clean_state_dict(resume["model_state_dict"]))\n    # Broadcast rank 0\'s weights *and* buffers so every rank starts identical.\n    # Buffers matter for transfer learning: batch-norm running statistics on the\n    # non-zero ranks would otherwise stay at their default (mean 0, var 1).\n    if distributed:\n        for parameter in base.parameters():\n            dist.broadcast(parameter.data, src=0)\n        for buffer in base.buffers():\n            dist.broadcast(buffer.data, src=0)\n\n    # Build the EMA from the plain module, before torch.compile and DDP wrap it.\n    ema = ModelEma(base, decay=args.ema_decay) if (args.ema_decay > 0 and rank == 0) else None\n\n    if args.compile:\n        try:\n            base = torch.compile(base)\n            if rank == 0:\n                print("torch.compile enabled.", flush=True)\n        except Exception as exc:\n            print(f"torch.compile unavailable, continuing eagerly: {exc}", flush=True)\n\n    model = (\n        DDP(base, device_ids=[local_rank], output_device=local_rank,\n            broadcast_buffers=True, find_unused_parameters=False)\n        if distributed else base\n    )\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    # A disabled scaler still needs a valid device type, so name CPU explicitly\n    # when there is no accelerator rather than constructing a CUDA object.\n    try:\n        scaler = torch.amp.GradScaler("cuda" if use_cuda else "cpu", enabled=use_cuda)\n    except (TypeError, RuntimeError):  # torch < 2.3 positional signature\n        scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)\n\n    steps_per_epoch = max(len(train_ds) // (args.local_batch_size * world), 1)\n    total_steps = steps_per_epoch * args.epochs\n    warmup_steps = max(int(total_steps * args.warmup_fraction), 1)\n\n    def lr_lambda(step: int) -> float:\n        if step < warmup_steps:\n            return (step + 1) / warmup_steps\n        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)\n        cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))\n        return args.min_lr_factor + (1.0 - args.min_lr_factor) * cosine\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n\n    start_epoch, history, best_f1, best_epoch, stale, prior_seconds = 1, [], -1.0, 0, 0, 0.0\n    global_step = 0\n    if resume is not None:\n        loadable = {k: v for k, v in resume.items() if k not in {"model_state_dict", "resume_config"}}\n        optimizer.load_state_dict(loadable["optimizer_state_dict"])\n        scaler.load_state_dict(loadable["scaler_state_dict"])\n        if "scheduler_state_dict" in loadable:\n            scheduler.load_state_dict(loadable["scheduler_state_dict"])\n        if ema is not None and loadable.get("ema_state_dict"):\n            ema.module.load_state_dict(loadable["ema_state_dict"])\n            ema.updates = int(loadable.get("ema_updates", 0))\n        start_epoch = int(resume["epoch"]) + 1\n        history = resume.get("history", []) if rank == 0 else []\n        best_f1 = float(resume.get("best_val_macro_f1", -1.0))\n        best_epoch = int(resume.get("best_epoch", 0))\n        stale = int(resume.get("stale", 0))\n        prior_seconds = float(resume.get("elapsed_seconds", 0.0))\n        global_step = int(resume.get("global_step", 0))\n        if stale >= args.patience:\n            start_epoch = args.epochs + 1\n            if rank == 0:\n                print("Early stopping was already reached; using the saved best checkpoint.", flush=True)\n\n    if rank == 0:\n        gpu_label = (\n            ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))\n            if use_cuda else "cpu"\n        )\n        print(\n            f"{args.arch} | devices={gpu_label} | "\n            f"global batch={args.local_batch_size * world} ({args.local_batch_size}/GPU) | "\n            f"workers/rank={args.workers} | lr={args.lr} | train={len(train_ds):,} | "\n            f"val={len(val_ds):,} | classes={len(classes)} | steps/epoch={steps_per_epoch}",\n            flush=True,\n        )\n        print(f"Dataset fingerprint: {fingerprint[:16]}...", flush=True)\n\n    started = time.time()\n    criterion = nn.CrossEntropyLoss(weight=loss_weight, label_smoothing=args.label_smoothing)\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        rng = np.random.default_rng(args.seed + epoch)\n        if args.shuffle_mode == "blocked":\n            order = block_shuffled_indices(len(train_ds), rng)\n        else:\n            order = rng.permutation(len(train_ds))\n        rank_indices = order[rank::world].tolist()\n\n        train_loader = DataLoader(\n            Subset(train_ds, rank_indices),\n            batch_size=args.local_batch_size,\n            shuffle=False,\n            drop_last=True,\n            num_workers=args.workers,\n            pin_memory=use_cuda,\n            persistent_workers=args.workers > 0,\n            prefetch_factor=4 if args.workers > 0 else None,\n        )\n        model.train()\n        epoch_loss = torch.zeros((), device=device)\n        images_seen = 0\n        epoch_started = time.perf_counter()\n        progress = tqdm(\n            total=len(train_loader),\n            desc=f"{args.arch} epoch {epoch}/{args.epochs} train",\n            unit="batch",\n            leave=True,\n            mininterval=1.0,\n            disable=(rank != 0),\n        )\n        for step, (images, labels) in enumerate(train_loader, 1):\n            x = gpu_augment_and_normalize(\n                images, device, training=True, mean=mean, std=std, rotation_deg=args.rotation_deg\n            )\n            y = labels.to(device=device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with amp_context(use_cuda):\n                logits = model(x)\n                loss = criterion(logits, y)\n            scaler.scale(loss).backward()\n            if args.grad_clip > 0:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            global_step += 1\n            if ema is not None:\n                ema.update(unwrap(model)._orig_mod if hasattr(unwrap(model), "_orig_mod") else unwrap(model))\n            epoch_loss += loss.detach()\n            images_seen += len(y)\n            if rank == 0:\n                progress.update(1)\n                if step % 20 == 0 or step == len(train_loader):\n                    elapsed = max(time.perf_counter() - epoch_started, 1e-6)\n                    progress.set_postfix(\n                        loss=f"{loss.detach().item():.4f}",\n                        avg_loss=f"{(epoch_loss.detach().item() / step):.4f}",\n                        img_s=f"{images_seen * world / elapsed:.1f}",\n                        lr=f"{scheduler.get_last_lr()[0]:.2e}",\n                        refresh=True,\n                    )\n            if args.dry_run and step >= 3:\n                break\n        progress.close()\n        if distributed:\n            dist.all_reduce(epoch_loss, op=dist.ReduceOp.SUM)\n        synchronise()\n\n        if args.dry_run:\n            if rank == 0:\n                print("Dry run complete: 3 steps executed, no checkpoint written.", flush=True)\n            synchronise()\n            if distributed:\n                dist.destroy_process_group()\n            return\n\n        report = [None]\n        if rank == 0:\n            train_loss = float(epoch_loss.detach().item() / max(len(train_loader) * world, 1))\n\n            raw_metrics, y_true, y_pred, _ = evaluate(\n                unwrap(model), val_ds, device, mean, std, args.eval_batch_size, args.workers, use_cuda\n            )\n            chosen_metrics, chosen_source = raw_metrics, "raw"\n            if ema is not None:\n                ema_metrics, ema_true, ema_pred, _ = evaluate(\n                    ema.module, val_ds, device, mean, std, args.eval_batch_size, args.workers, use_cuda\n                )\n                if ema_metrics["macro_f1"] > raw_metrics["macro_f1"]:\n                    chosen_metrics, chosen_source = ema_metrics, "ema"\n                    y_true, y_pred = ema_true, ema_pred\n\n            # Record the applied learning rate. Without this the schedule is\n            # invisible in the artifacts: protocol v1 decayed to nearly zero\n            # every run and nothing on disk said so.\n            epoch_lr = scheduler.get_last_lr()[0] if scheduler is not None else args.lr\n\n            row = {\n                "model": args.arch,\n                "epoch": epoch,\n                "lr": epoch_lr,\n                "train_loss": train_loss,\n                **{f"val_{key}": value for key, value in raw_metrics.items()},\n                f"val_macro_f1_{chosen_source}": chosen_metrics["macro_f1"],\n                "weights_used": chosen_source,\n                "elapsed_minutes": round((prior_seconds + time.time() - started) / 60.0, 2),\n            }\n            history.append(row)\n\n            if chosen_metrics["macro_f1"] > best_f1:\n                best_f1, best_epoch, stale = chosen_metrics["macro_f1"], epoch, 0\n                state = ema.module.state_dict() if chosen_source == "ema" else unwrap(model).state_dict()\n                atomic_save(\n                    {\n                        "architecture": args.arch,\n                        "model_state_dict": clean_state_dict(state),\n                        "classes": classes,\n                        "image_size": args.image_size,\n                        "pretrained": True,\n                        "normalization_mean": list(mean),\n                        "normalization_std": list(std),\n                        "epoch": epoch,\n                        "val_macro_f1": best_f1,\n                        "weights_are_ema": chosen_source == "ema",\n                        "seed": args.seed,\n                        "dataset_fingerprint": fingerprint,\n                        "n_train": len(train_ds),\n                        "n_val": len(val_ds),\n                    },\n                    best_path,\n                )\n                pd.DataFrame(classification_table(y_true, y_pred, classes)).to_csv(\n                    output_dir / "validation_classification_report.csv", index=False\n                )\n                pd.DataFrame(\n                    confusion_matrix(y_true, y_pred, labels=np.arange(len(classes))),\n                    index=classes,\n                    columns=classes,\n                ).to_csv(output_dir / "validation_confusion_matrix.csv")\n                (output_dir / "validation_metrics.json").write_text(\n                    json.dumps({**chosen_metrics, "weights_used": chosen_source, "epoch": epoch}, indent=2),\n                    encoding="utf-8",\n                )\n            else:\n                stale += 1\n\n            elapsed_seconds = prior_seconds + time.time() - started\n            atomic_save(\n                {\n                    "architecture": args.arch,\n                    "model_state_dict": clean_state_dict(unwrap(model).state_dict()),\n                    "optimizer_state_dict": optimizer.state_dict(),\n                    "scaler_state_dict": scaler.state_dict(),\n                    "scheduler_state_dict": scheduler.state_dict(),\n                    "ema_state_dict": clean_state_dict(ema.module.state_dict()) if ema is not None else None,\n                    "ema_updates": ema.updates if ema is not None else 0,\n                    "classes": classes,\n                    "epoch": epoch,\n                    "best_val_macro_f1": best_f1,\n                    "best_epoch": best_epoch,\n                    "stale": stale,\n                    "history": history,\n                    "global_step": global_step,\n                    "elapsed_seconds": elapsed_seconds,\n                    "dataset_fingerprint": fingerprint,\n                    "resume_config": resume_config,\n                },\n                last_path,\n            )\n            pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)\n            print(\n                f"Epoch {epoch}/{args.epochs} | loss={train_loss:.4f} | "\n                f"val_acc={raw_metrics[\'accuracy\']:.4f} | val_macro_f1={raw_metrics[\'macro_f1\']:.4f} | "\n                f"val_f1_used={chosen_metrics[\'macro_f1\']:.4f} ({chosen_source}) | "\n                f"ece={raw_metrics.get(\'ece\', float(\'nan\')):.4f} | best_epoch={best_epoch} | "\n                f"elapsed={elapsed_seconds / 60.0:.2f} min",\n                flush=True,\n            )\n            report[0] = {\n                "best_f1": best_f1,\n                "best_epoch": best_epoch,\n                "stale": stale,\n                "stop": stale >= args.patience,\n                "elapsed_seconds": elapsed_seconds,\n            }\n        if distributed:\n            dist.broadcast_object_list(report, src=0, device=device)\n        best_f1, best_epoch, stale = report[0]["best_f1"], report[0]["best_epoch"], report[0]["stale"]\n        prior_seconds = report[0]["elapsed_seconds"]\n        if report[0]["stop"]:\n            if rank == 0:\n                print(f"Early stopping: validation macro-F1 did not improve for {args.patience} epochs.", flush=True)\n            break\n\n    synchronise()\n    if rank == 0:\n        if not best_path.is_file():\n            raise FileNotFoundError("No best checkpoint was written; training never reached a validation pass.")\n        best = safe_load(best_path, device)\n        final = build_model(args.arch, len(classes), pretrained=False).to(device)\n        final.load_state_dict(clean_state_dict(best["model_state_dict"]))\n        final.eval()\n        final_metrics, y_true, y_pred, _ = evaluate(final, val_ds, device, mean, std, args.eval_batch_size, args.workers, use_cuda)\n        summary = {\n            "model": args.arch,\n            "pretrained": bool(pretrained),\n            "validation_images": len(val_ds),\n            "train_images": len(train_ds),\n            **{f"val_{key}": value for key, value in final_metrics.items()},\n            "best_epoch": best_epoch,\n            "weights_are_ema": bool(best.get("weights_are_ema")),\n            "parameters": sum(p.numel() for p in final.parameters()),\n            "checkpoint_mb": round(best_path.stat().st_size / (1024 ** 2), 2),\n            "train_minutes": round(prior_seconds / 60.0, 2),\n            "training_mode": "imagenet_finetune_ddp_v2",\n            "used_gpus": world,\n            "local_batch_size": args.local_batch_size,\n            "batch_size": args.local_batch_size * world,\n            "global_batch_size": args.local_batch_size * world,\n            "workers_per_rank": args.workers,\n            "learning_rate": args.lr,\n            "weight_decay": args.weight_decay,\n            "rotation_deg": args.rotation_deg,\n            "warmup_fraction": args.warmup_fraction,\n            "min_lr_factor": args.min_lr_factor,\n            "max_epochs": args.epochs,\n            "patience": args.patience,\n            "label_smoothing": args.label_smoothing,\n            "class_weight_alpha": args.class_weight_alpha,\n            "ema_decay": args.ema_decay,\n            "checkpoint": str(best_path),\n            "dataset_fingerprint": fingerprint,\n        }\n        (output_dir / "validation_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n        print("Training complete:", json.dumps(summary, indent=2), flush=True)\n    synchronise()\n    if distributed:\n        dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/sweep_worker.py': '#!/usr/bin/env python3\n"""Run one architecture over several seeds, on either one GPU or both.\n\nThis exists so a sweep can be scheduled instead of run one-architecture-per-\nsession. It shells out to ``train_ddp.py`` once per seed, which keeps every run\nan independent process with its own resumable checkpoint.\n\nTwo modes:\n\n``--mode ddp``\n    Uses both GPUs through ``torch.distributed.run``. Right for the models that\n    are bandwidth-bound on a single T4 (EfficientNet-B0, ConvNeXt-Tiny).\n\n``--mode single``\n    One process, no process group, pinned with ``CUDA_VISIBLE_DEVICES``. Right\n    for the small models, where two GPUs per model waste half the machine. Two\n    of these can run side by side, one per GPU.\n\nWhy the split matters: the pilot measured SmallCNN at 0.67 min/epoch and\nEfficientNet-B0 at 2.45 min/epoch. Running SmallCNN on both GPUs to finish in\n40 seconds is worse than running it on one GPU while something else uses the\nother.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nENGINE = HERE / "train_ddp.py"\n\n\ndef build_command(arch, seed, cfg, protocol, paths, out_dir, mode, script, device, epochs):\n    # Arguments only. The interpreter and the script path are added once, by the\n    # mode branch below: torchrun takes the script as its first positional\n    # argument, so leaving "python" in here made torchrun try to execute the\n    # interpreter binary as Python source ("source code cannot contain null\n    # bytes" / "ELF" in the traceback).\n    args = [\n        "--arch", arch,\n        "--train-images", str(paths["train_x"]), "--train-labels", str(paths["train_y"]),\n        "--val-images", str(paths["val_x"]), "--val-labels", str(paths["val_y"]),\n        "--classes-json", str(paths["classes"]), "--output-dir", str(out_dir),\n        "--epochs", str(epochs), "--patience", str(protocol["patience"]),\n        "--local-batch-size", str(cfg["local_batch_size"]),\n        "--eval-batch-size", str(protocol["eval_batch_size"]),\n        "--lr", str(cfg["lr"]), "--weight-decay", str(cfg["weight_decay"]),\n        "--workers", str(protocol["workers"]),\n        "--label-smoothing", str(protocol["label_smoothing"]),\n        "--class-weight-alpha", str(protocol["class_weight_alpha"]),\n        "--ema-decay", str(protocol["ema_decay"]),\n        "--grad-clip", str(protocol["grad_clip"]),\n        "--warmup-fraction", str(protocol["warmup_fraction"]),\n        "--min-lr-factor", str(protocol.get("min_lr_factor", 0.01)),\n        "--rotation-deg", str(protocol["rotation_deg"]),\n        "--shuffle-mode", protocol["shuffle_mode"],\n        "--image-size", str(protocol["image_size"]),\n        "--seed", str(seed),\n    ]\n    if mode == "ddp":\n        # torchrun\'s first positional argument must be the training script.\n        return [sys.executable, "-m", "torch.distributed.run", "--standalone",\n                "--nnodes=1", "--nproc_per_node=2", str(script)] + args\n    # Single process on whichever GPU CUDA_VISIBLE_DEVICES exposes as 0.\n    return [sys.executable, str(script)] + args + ["--no-dist", "--device", device]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--arch", required=True)\n    parser.add_argument("--seeds", nargs="+", type=int, required=True)\n    parser.add_argument("--mode", choices=("ddp", "single"), required=True)\n    parser.add_argument("--protocol", type=Path, required=True)\n    parser.add_argument("--train-images", type=Path, required=True)\n    parser.add_argument("--train-labels", type=Path, required=True)\n    parser.add_argument("--val-images", type=Path, required=True)\n    parser.add_argument("--val-labels", type=Path, required=True)\n    parser.add_argument("--classes-json", type=Path, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--engine", type=Path, default=ENGINE)\n    parser.add_argument("--device", default="cuda", choices=("auto", "cuda", "cpu"),\n                        help="single-gpu mode only; cpu exists for local tests")\n    parser.add_argument("--epochs", type=int, default=None,\n                        help="override protocol epochs; for fast local checks")\n    parser.add_argument("--verbose", action="store_true",\n                        help="echo each engine command before running it")\n    args = parser.parse_args()\n\n    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))\n    if args.arch not in protocol["architectures"]:\n        raise SystemExit(f"{args.arch} is not in protocol.json")\n    cfg = protocol["architectures"][args.arch]\n    paths = {\n        "train_x": args.train_images, "train_y": args.train_labels,\n        "val_x": args.val_images, "val_y": args.val_labels,\n        "classes": args.classes_json,\n    }\n\n    print(f"[{args.arch}] mode={args.mode} seeds={args.seeds} "\n          f"gpus={os.environ.get(\'CUDA_VISIBLE_DEVICES\', \'all\')}", flush=True)\n\n    failures = []\n    for seed in args.seeds:\n        out_dir = args.output_root / args.arch / f"seed{seed}"\n        if (out_dir / "validation_metrics.json").is_file():\n            print(f"[{args.arch} seed {seed}] already complete, skipping", flush=True)\n            continue\n        out_dir.mkdir(parents=True, exist_ok=True)\n        epochs = args.epochs if args.epochs is not None else protocol["epochs"]\n        command = build_command(args.arch, seed, cfg, protocol, paths, out_dir,\n                                args.mode, args.engine, args.device, epochs)\n        if args.verbose:\n            print("  " + " ".join(command), flush=True)\n        started = time.time()\n        result = subprocess.run(command, env={**os.environ, "PYTHONUNBUFFERED": "1"}, check=False)\n        minutes = (time.time() - started) / 60.0\n        status = "ok" if result.returncode == 0 else f"FAILED ({result.returncode})"\n        print(f"[{args.arch} seed {seed}] {status} in {minutes:.2f} min", flush=True)\n        if result.returncode != 0:\n            failures.append(seed)\n            # Echo the command so a failure is diagnosable from the log alone.\n            print("  command: " + " ".join(command), flush=True)\n\n    if failures:\n        raise SystemExit(f"[{args.arch}] seeds failed: {failures}")\n    print(f"[{args.arch}] all requested seeds complete", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/protocol_v2.json': '{\n  "version": 2,\n  "supersedes": "configs/protocol.json (v1). v1 was measured to be invalid on 2026-10-25: the backbone never fine-tuned, EfficientNet-B0 reached 0.6815 validation accuracy against 0.9954 for the reference. Evidence: reports/model_comparison/SWEEP_V1_INVALID.md",\n  "note": "Restores the optimisation settings that actually produced the reference checkpoints, instead of a modern recipe that was never validated. Every value below is traceable to a source: efficientnet_b0 to notebooks/01_Train_EfficientNet_B0.ipynb (the script that produced the 99.54% checkpoint), smallcnn to reports/phase2_baseline/experiment_config.json (96.01%). resnet18 and convnext_tiny have no recorded original config, so they inherit the validated efficientnet_b0 recipe.",\n  "image_size": 224,\n  "epochs": 12,\n  "patience": 3,\n  "seeds": [\n    42,\n    43,\n    44\n  ],\n  "eval_batch_size": 128,\n  "workers": 4,\n  "shuffle_mode": "random",\n  "rotation_deg": 12.0,\n  "grad_clip": 5.0,\n  "warmup_fraction": 0.0,\n  "min_lr_factor": 1.0,\n  "label_smoothing": 0.0,\n  "class_weight_alpha": 0.0,\n  "ema_decay": 0.0,\n  "selection_metric": "val_macro_f1",\n  "architectures": {\n    "smallcnn": {\n      "pretrained": false,\n      "local_batch_size": 32,\n      "lr": 0.001,\n      "weight_decay": 0.0001,\n      "why": "trains from scratch; values copied from reports/phase2_baseline/experiment_config.json (global batch 64 = 32 per GPU, AdamW lr 1e-3, weight decay 1e-4), which produced 96.01% validation accuracy. This is the only sanctioned deviation from the shared recipe."\n    },\n    "resnet18": {\n      "pretrained": true,\n      "local_batch_size": 16,\n      "lr": 0.0001,\n      "weight_decay": 0.0001,\n      "why": "no original config is on record for the 96.15% ResNet18 checkpoint, so it inherits the validated efficientnet_b0 recipe unchanged rather than a guessed one."\n    },\n    "efficientnet_b0": {\n      "pretrained": true,\n      "local_batch_size": 16,\n      "lr": 0.0001,\n      "weight_decay": 0.0001,\n      "why": "copied from notebooks/01_Train_EfficientNet_B0.ipynb: AdamW lr 1e-4, weight decay 1e-4, global batch 32 (16 per GPU), constant learning rate, no label smoothing, no class weights. This produced the 99.54% reference checkpoint, reaching 98.26% validation accuracy in epoch 1."\n    },\n    "convnext_tiny": {\n      "pretrained": true,\n      "local_batch_size": 16,\n      "lr": 0.0001,\n      "weight_decay": 0.0001,\n      "why": "no original config on record; inherits the validated efficientnet_b0 recipe. Its 2018-era-era activation cost is higher, but at 16 per GPU it fits, and keeping the recipe identical is what makes the cross-architecture check meaningful."\n    }\n  },\n  "notes": {\n    "constant_lr": "warmup_fraction 0.0 with min_lr_factor 1.0 makes the schedule flat: lr_lambda returns 1.0 for every step. This reproduces the constant learning rate of the original run. Setting either value back to its v1 value re-introduces decay.",\n    "what_v1_got_wrong": "v1 used weight decay 0.05 (500x the proven 1e-4), global batch 128 (4x fewer optimizer steps than the proven 32), warmup plus cosine decay to near zero (the proven run held the rate constant), and label smoothing 0.05 with class weights (the proven run used neither).",\n    "class_balancing": "Class-weighted loss and label smoothing are NOT part of the baseline arm. They belong to the H3 treatment arm and are introduced there, separately, so that if they change the result we know it was them.",\n    "augmentation": "Horizontal flip, rotation up to 12 degrees, brightness 0.88-1.12 and per-channel gain 0.92-1.08, applied on the GPU inside the engine. The original run used flip, rotation and brightness; the per-channel gain is the only addition, and it is kept because intensity variation is one of the documented differences between PlantVillage and field images.",\n    "gate": "No sweep may run until one 2-epoch EfficientNet-B0 run reaches 0.95 validation accuracy by epoch 2. The original reached 0.9826 in epoch 1, so the gate is deliberately loose: it detects a broken configuration, not a slightly weaker one.",\n    "shuffle": "v2 initially inherited shuffle_mode \'blocked\' from v1. Measured on the real manifest (scripts/check_batch_purity.py): the manifest is written in class order, so permuting 2048-image blocks and shuffling inside them yields batches with a median of 3 distinct classes, against 12 under a full random shuffle. That is a BatchNorm problem: a batch drawn from three classes produces class-specific normalisation statistics. It also explains the v1 sweep pattern, where the only architecture that trained (ConvNeXt, 0.95) uses LayerNorm and the three that collapsed (EfficientNet-B0 0.68, ResNet18 0.58, SmallCNN 0.23) all use BatchNorm. Random shuffle removes the effect. The original run that reached 0.9826 in one epoch used an ordinary shuffled DataLoader."\n  }\n}\n',
    '/kaggle/working/make_leaf_masks.py': '#!/usr/bin/env python3\n"""Separate leaf from background for every PlantVillage training image.\n\nNeeded by S19: to composite a leaf onto a field background, the leaf has to be\nisolated first.\n\nTwo cues are combined because neither is sufficient alone.\n\n**Corner colour.** PlantVillage backgrounds are close to uniform, so the four\ncorners give a good estimate of the backdrop. Pixels far from that colour in RGB\nare foreground. This fails on a leaf that matches the backdrop in colour.\n\n**Greenness.** An HSV hue window catches leaf tissue even when it resembles the\nbackdrop, but misses brown or senescent lesions, which are not green.\n\nThe union of the two is used. That is a heuristic and is stated as one: this is\nnot a trained segmentation model, and its errors are visible in the rendered\ncheck sheet the script writes.\n\nOutput is one packed bit array per split (`np.packbits`), roughly 230 MB for\n36,762 images instead of 1.8 GB as uint8.\n\n    python scripts/make_leaf_masks.py --split train --out data/derived/masks_train.npz\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parents[1]\nSIZE = 224\n\n\ndef leaf_mask(array: np.ndarray, distance: float = 60.0) -> np.ndarray:\n    """Boolean foreground mask for one uint8 RGB image of shape (SIZE, SIZE, 3)."""\n    corners = np.concatenate([\n        array[:12, :12].reshape(-1, 3), array[:12, -12:].reshape(-1, 3),\n        array[-12:, :12].reshape(-1, 3), array[-12:, -12:].reshape(-1, 3),\n    ])\n    backdrop = np.median(corners, axis=0)\n    far_from_backdrop = np.linalg.norm(array.astype(np.float32) - backdrop, axis=2) > distance\n\n    hsv = np.asarray(Image.fromarray(array).convert("HSV"), dtype=np.int16)\n    hue, sat, val = hsv[..., 0], hsv[..., 1], hsv[..., 2]\n    green = (hue >= 25) & (hue <= 105) & (sat >= 40) & (val >= 25)\n\n    mask = far_from_backdrop | green\n    return mask\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--split", default="train", choices=("train", "validation"))\n    parser.add_argument("--image-root", type=Path,\n                        default=ROOT / "data" / "raw" / "PlantVillage" / "train")\n    parser.add_argument("--manifest", type=Path,\n                        default=ROOT / "data" / "manifests" / "plantvillage_development.csv")\n    parser.add_argument("--out", type=Path, default=ROOT / "data" / "derived" / "masks_train.npz")\n    parser.add_argument("--check-sheet", type=Path,\n                        default=ROOT / "reports" / "mechanism" / "mask_check.png")\n    args = parser.parse_args()\n\n    rows = []\n    with args.manifest.open(encoding="utf-8") as handle:\n        for row in csv.DictReader(handle):\n            if row["split"] == args.split:\n                rows.append(row["relative_path"])\n    if not rows:\n        raise SystemExit(f"no {args.split} rows in {args.manifest}")\n\n    masks = np.zeros((len(rows), SIZE, SIZE), dtype=bool)\n    started = time.perf_counter()\n    coverage = []\n    for i, relative in enumerate(rows):\n        with Image.open(args.image_root / relative) as image:\n            array = np.asarray(image.convert("RGB").resize((SIZE, SIZE), Image.Resampling.BILINEAR))\n        mask = leaf_mask(array)\n        masks[i] = mask\n        coverage.append(mask.mean())\n        if (i + 1) % 5000 == 0:\n            print(f"  {i + 1:,}/{len(rows):,}  {time.perf_counter() - started:.0f}s", flush=True)\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    packed = np.packbits(masks.reshape(len(rows), -1), axis=1)\n    np.savez_compressed(args.out,\n                        packed=packed, n=len(rows), size=SIZE, split=args.split,\n                        paths=np.array(rows))\n\n    coverage = np.array(coverage)\n    summary = {\n        "split": args.split,\n        "images": len(rows),\n        "mask_coverage_mean": round(float(coverage.mean()), 4),\n        "mask_coverage_p05": round(float(np.percentile(coverage, 5)), 4),\n        "mask_coverage_p95": round(float(np.percentile(coverage, 95)), 4),\n        "images_under_5pct_foreground": int((coverage < 0.05).sum()),\n        "images_over_95pct_foreground": int((coverage > 0.95).sum()),\n        "method": "corner-colour distance OR HSV green window; heuristic, not a segmentation model",\n        "elapsed_seconds": round(time.perf_counter() - started, 1),\n        "output": str(args.out),\n    }\n    print(json.dumps(summary, indent=2))\n\n    # Visual check: the failure modes above are invisible in a coverage number.\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    picks = list(np.argsort(coverage))[:2] + list(np.argsort(coverage)[-2:]) + list(range(2))\n    fig, axes = plt.subplots(len(picks), 2, figsize=(6, 3 * len(picks)))\n    for row_i, index in enumerate(picks):\n        with Image.open(args.image_root / rows[index]) as image:\n            array = np.asarray(image.convert("RGB").resize((SIZE, SIZE), Image.Resampling.BILINEAR))\n        axes[row_i][0].imshow(array)\n        axes[row_i][0].set_title(f"coverage {coverage[index]:.2f}", fontsize=8)\n        axes[row_i][0].axis("off")\n        axes[row_i][1].imshow(masks[index], cmap="gray")\n        axes[row_i][1].set_title("mask", fontsize=8)\n        axes[row_i][1].axis("off")\n    fig.suptitle("Leaf mask spot check: lowest and highest coverage, plus a typical case", fontsize=9)\n    fig.tight_layout()\n    args.check_sheet.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(args.check_sheet, dpi=140)\n    plt.close(fig)\n    print(f"wrote {args.check_sheet}")\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/build_composite_cache.py': '#!/usr/bin/env python3\n"""Composite PlantVillage leaves onto field backgrounds, for S19.\n\nThe treatment arm for H3. Every training image keeps its leaf and its label, and\nloses its laboratory backdrop. The model then has no consistent laboratory context\nto lean on, which is the whole point of the experiment.\n\nDesign decisions, stated because they bound what the result can mean:\n\n**One random background per image, fixed for the run.** Re-sampling every epoch\nwould be better augmentation but needs the engine to hold a background pool and a\nper-image mask during training, which is a larger change to a working pipeline.\nWith a fixed random assignment the mapping from background to class is already\ndestroyed, because each class receives random backdrops. What is lost is\nbackground diversity per image. That limitation goes in the report.\n\n**Validation is not composited.** The PlantVillage validation score has to stay\ncomparable to the baseline, so only the training cache is altered. Field\nevaluation on PlantDoc is untouched either way.\n\n**The background pool is sampled once and recorded.** `background_index.npy`\nstores which background each image received, so the treatment is reproducible.\n\n    python scripts/build_composite_cache.py \\\n        --backgrounds data/raw/DeepWeeds \\\n        --masks data/derived/masks_train.npz \\\n        --train-cache /kaggle/temp/plantvillage_cache/train_224_...\\\n        --out /kaggle/temp/plantvillage_cache/train_composite_224_x.npy\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parents[1]\nSIZE = 224\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n\n\ndef load_pool(directory: Path, limit: int) -> np.ndarray:\n    """Read up to `limit` background images, resized to the training size."""\n    files = [p for p in sorted(directory.rglob("*"))\n             if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]\n    if not files:\n        raise SystemExit(f"no images under {directory}")\n    if limit:\n        files = files[:limit]\n    pool = np.zeros((len(files), SIZE, SIZE, 3), dtype=np.uint8)\n    unreadable = 0\n    for i, path in enumerate(files):\n        try:\n            with Image.open(path) as image:\n                pool[i] = np.asarray(image.convert("RGB").resize((SIZE, SIZE), Image.Resampling.BILINEAR))\n        except Exception:\n            unreadable += 1\n    if unreadable:\n        print(f"  {unreadable} background(s) unreadable, left black")\n    print(f"background pool: {len(files)} images from {directory}")\n    return pool\n\n\ndef unpack_masks(path: Path) -> np.ndarray:\n    blob = np.load(path, allow_pickle=False)\n    packed, n, size = blob["packed"], int(blob["n"]), int(blob["size"])\n    if size != SIZE:\n        raise SystemExit(f"masks were built at {size}px, expected {SIZE}")\n    flat = np.unpackbits(packed, axis=1)[:, : size * size]\n    return flat.reshape(n, size, size).astype(bool)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--backgrounds", type=Path, required=True)\n    parser.add_argument("--masks", type=Path, required=True)\n    parser.add_argument("--train-cache", type=Path, required=True,\n                        help="the uint8 memmap the engine already trains on")\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--pool-limit", type=int, default=4000)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--limit", type=int, default=0, help="stop early, for a smoke test")\n    parser.add_argument("--min-fg", type=float, default=0.25,\n                        help="below this foreground fraction the mask is unreliable")\n    parser.add_argument("--max-fg", type=float, default=0.90,\n                        help="above this the mask is unreliable, usually a full-frame texture")\n    parser.add_argument("--check-sheet", type=Path, default=None)\n    args = parser.parse_args()\n\n    pool = load_pool(args.backgrounds, args.pool_limit)\n    masks = unpack_masks(args.masks)\n    source = np.load(args.train_cache, mmap_mode="r", allow_pickle=False)\n\n    n = min(len(source), len(masks), args.limit or len(source))\n    if len(masks) != len(source):\n        raise SystemExit(\n            f"mask count {len(masks)} does not match cache rows {len(source)}. "\n            "Masks must be built from the same split in the same order."\n        )\n    print(f"compositing {n:,} training images over a pool of {len(pool)} backgrounds")\n\n    out = np.lib.format.open_memmap(args.out, mode="w+", dtype=np.uint8,\n                                    shape=(n, SIZE, SIZE, 3))\n    rng = np.random.default_rng(args.seed)\n    chosen = rng.integers(0, len(pool), size=n)\n    started = time.perf_counter()\n\n    # The corner-colour plus greenness heuristic fails on close-up leaf textures,\n    # where the frame is almost entirely leaf. There the mask can claim 0.08 or\n    # 1.00 foreground, and compositing would replace leaf tissue with field\n    # background. Those images pass through unaltered rather than being trained\n    # on in a corrupted form.\n    coverage = masks[:n].mean(axis=(1, 2))\n    trusted = (coverage >= args.min_fg) & (coverage <= args.max_fg)\n    print(f"  trusted masks: {trusted.sum():,}/{n:,} ({trusted.mean():.1%}); "\n          f"{n - trusted.sum():,} pass through unaltered")\n\n    for i in range(n):\n        image = np.asarray(source[i])\n        if not trusted[i]:\n            out[i] = image\n            continue\n        back = pool[chosen[i]]\n        keep = masks[i][..., None]\n        out[i] = np.where(keep, image, back)\n        if (i + 1) % 5000 == 0:\n            rate = (i + 1) / max(time.perf_counter() - started, 1e-6)\n            print(f"  {i + 1:,}/{n:,}  {rate:.0f} img/s", flush=True)\n    out.flush()\n    del out\n\n    np.save(Path(str(args.out)).with_suffix(".background_index.npy"), chosen)\n    summary = {\n        "images": n,\n        "backgrounds_in_pool": len(pool),\n        "backgrounds_directory": str(args.backgrounds),\n        "masks": str(args.masks),\n        "source_cache": str(args.train_cache),\n        "output": str(args.out),\n        "seed": args.seed,\n        "policy": "one random background per image, fixed for the run; validation not composited",\n        "mean_foreground_fraction": round(float(masks[:n].mean()), 4),\n        "images_composited": int(trusted.sum()),\n        "images_passed_through": int(n - trusted.sum()),\n        "mask_trust_band": [args.min_fg, args.max_fg],\n        "pass_through_reason": ("HSV and corner-colour masks are unreliable outside the trust "\n                                "band, typically on close-up leaf textures; those images keep "\n                                "their original background"),\n        "elapsed_seconds": round(time.perf_counter() - started, 1),\n    }\n    Path(str(args.out) + ".json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n\n    if args.check_sheet:\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n        pool_idx = np.flatnonzero(trusted)\n        skip_idx = np.flatnonzero(~trusted)\n        picks = list(pool_idx[:4]) + list(skip_idx[:2])\n        if len(picks) < 6:                      # pad so no panel is left empty\n            picks += [i for i in pool_idx[4:] if i not in picks][: 6 - len(picks)]\n        if not picks:\n            picks = list(rng.choice(n, size=min(6, n), replace=False))\n        fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))\n        for col, index in enumerate(picks):\n            axes[0][col].imshow(np.asarray(source[index]))\n            axes[0][col].set_title("before", fontsize=7)\n            axes[1][col].imshow(np.asarray(np.load(args.out, mmap_mode="r", allow_pickle=False)[index]))\n            axes[1][col].set_title("after", fontsize=7)\n            for row in range(2):\n                axes[row][col].axis("off")\n        fig.suptitle("Composite check: leaf kept, laboratory backdrop replaced", fontsize=10)\n        fig.tight_layout()\n        Path(args.check_sheet).parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(args.check_sheet, dpi=140)\n        plt.close(fig)\n        print(f"wrote {args.check_sheet}")\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/check_background_disjointness.py': '#!/usr/bin/env python3\n"""Prove a background corpus does not overlap PlantDoc (roadmap S18).\n\nH3 hangs on this check. If a single augmentation background came from PlantDoc,\nthe field evaluation would be seeing training data and the −78.4 pp result would\nbe void. "They are different datasets" is not evidence; a hash comparison is.\n\n    python scripts/check_background_disjointness.py --backgrounds data/raw/DeepWeeds\n\nExit code 1 means overlap was found and training must not start.\n\nWhat this does and does not catch\n---------------------------------\nIt compares SHA-256 of every background file against the SHA-256 values the\nPlantDoc manifests already record per image, and against PlantVillage for good\nmeasure. That catches the same file, and any file byte-identical after\nre-encoding.\n\nIt does **not** catch a near-duplicate that was re-saved at a different quality,\ncropped, or resized. Catching that needs perceptual hashing, which is a different\ntool and is not implemented here. The output says so, so nobody reads a pass as\nstronger than it is.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nMANIFESTS = ROOT / "data" / "manifests"\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}\n\n\ndef sha256_of(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef manifest_hashes(names: list[str]) -> tuple[dict[str, str], int]:\n    """Collect the recorded sha256 values from the given manifest files."""\n    known, missing = {}, 0\n    for name in names:\n        path = MANIFESTS / name\n        if not path.is_file():\n            print(f"  warning: {name} not found, skipping")\n            continue\n        with path.open(encoding="utf-8") as handle:\n            for row in csv.DictReader(handle):\n                digest = (row.get("sha256") or "").strip().lower()\n                if digest:\n                    known[digest] = row.get("relative_path", "?")\n                else:\n                    missing += 1\n    return known, missing\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--backgrounds", type=Path, required=True,\n                        help="directory containing candidate background images")\n    parser.add_argument("--out", type=Path, default=ROOT / "reports" / "mechanism" / "disjointness.json")\n    args = parser.parse_args()\n\n    if not args.backgrounds.is_dir():\n        raise SystemExit(f"background directory not found: {args.backgrounds}")\n\n    plantdoc, pd_missing = manifest_hashes(["plantdoc_development.csv", "plantdoc_test.csv"])\n    plantvillage, pv_missing = manifest_hashes(["plantvillage_development.csv", "plantvillage_test.csv"])\n    print(f"PlantDoc hashes on record : {len(plantdoc):,} ({pd_missing} rows had none)")\n    print(f"PlantVillage hashes       : {len(plantvillage):,} ({pv_missing} rows had none)")\n\n    candidates = [p for p in args.backgrounds.rglob("*")\n                  if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]\n    if not candidates:\n        raise SystemExit(f"no images found under {args.backgrounds}")\n    print(f"candidate backgrounds     : {len(candidates):,}")\n\n    overlaps, unreadable = [], []\n    for i, path in enumerate(candidates, start=1):\n        try:\n            digest = sha256_of(path)\n        except OSError as exc:\n            unreadable.append((str(path), repr(exc)))\n            continue\n        if digest in plantdoc:\n            overlaps.append({"background": str(path.relative_to(args.backgrounds)),\n                             "collides_with": plantdoc[digest], "corpus": "PlantDoc"})\n        elif digest in plantvillage:\n            overlaps.append({"background": str(path.relative_to(args.backgrounds)),\n                             "collides_with": plantvillage[digest], "corpus": "PlantVillage"})\n        if i % 2000 == 0:\n            print(f"  hashed {i:,}/{len(candidates):,}", flush=True)\n\n    verdict = "FAIL: overlap found, do not train" if overlaps else "PASS: no byte-identical overlap"\n    payload = {\n        "backgrounds": str(args.backgrounds),\n        "candidate_images": len(candidates),\n        "plantdoc_hashes": len(plantdoc),\n        "plantvillage_hashes": len(plantvillage),\n        "overlaps": overlaps,\n        "unreadable": unreadable,\n        "verdict": verdict,\n        "limitation": ("exact SHA-256 only; a re-encoded, cropped or resized near-duplicate "\n                       "would not be caught, and no perceptual hashing was performed"),\n    }\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    args.out.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n\n    print(f"\\n{verdict}")\n    if overlaps:\n        for hit in overlaps[:20]:\n            print(f"  {hit[\'background\']} == {hit[\'collides_with\']} ({hit[\'corpus\']})")\n        if len(overlaps) > 20:\n            print(f"  ... and {len(overlaps) - 20} more")\n    print(f"wrote {args.out}")\n    if overlaps:\n        raise SystemExit(1)\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/evaluate_plantdoc.py': '#!/usr/bin/env python3\n"""Score a PlantVillage-trained checkpoint on PlantDoc and measure the field gap.\n\nThis is roadmap S11/S12: the number the whole project exists to produce. Every\nmodel so far has only ever seen PlantVillage, which is lab imagery. PlantDoc is\nfield photography - real soil, shadows, occlusion, camera angle - and nothing\nhere is ever trained or tuned on it.\n\nThree things make this honest rather than optimistic:\n\n**The mapped subset is the unit.** Only PlantDoc classes that resolve to a\nPlantVillage class are scored, and the count is printed. Predicting a\nPlantVillage class that PlantDoc has no counterpart for is counted separately as\nout-of-mapping, never silently marked wrong or silently dropped.\n\n**Macro-F1 is computed only over classes that actually appear.** A macro average\nover classes with zero support is meaningless; the earlier audit found this\nproduced a macro-F1 of 0.026 on a 96-image run.\n\n**Intervals come from a bootstrap over images**, because PlantDoc test is small\nand a point estimate without a width invites over-reading.\n\n    python scripts/evaluate_plantdoc.py \\\n        --checkpoint checkpoints/efficientnet_b0/efficientnet_b0_best.pt \\\n        --arch efficientnet_b0 --split test --out reports/field_gap/b0_test\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\n\nfrom plant_disease.models import build_model  # noqa: E402\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\nclass PlantDocImages(Dataset):\n    """Reads PlantDoc images and returns them normalised for an ImageNet model."""\n\n    def __init__(self, rows, image_root: Path, size: int):\n        self.rows = rows\n        self.image_root = image_root\n        self.size = size\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        path = self.image_root / row["relative_path"]\n        with Image.open(path) as image:\n            picture = image.convert("RGB").resize((self.size, self.size), Image.Resampling.BILINEAR)\n            array = np.asarray(picture, dtype=np.float32) / 255.0\n        tensor = torch.from_numpy(array).permute(2, 0, 1)\n        mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)\n        std = torch.tensor(IMAGENET_STD).view(3, 1, 1)\n        return (tensor - mean) / std, index\n\n\ndef load_map(path: Path):\n    """Returns {plantdoc_label: plantvillage_label}."""\n    with path.open(encoding="utf-8") as handle:\n        return {row["plantdoc_label"]: row["plantvillage_label"] for row in csv.DictReader(handle)}\n\n\ndef load_rows(manifest: Path, split: str, label_map: dict):\n    """Returns scored rows (label maps to PlantVillage) and the unmapped count."""\n    scored, unmapped = [], Counter()\n    with manifest.open(encoding="utf-8") as handle:\n        for row in csv.DictReader(handle):\n            if split != "all" and row["split"] != split:\n                continue\n            target = label_map.get(row["label"])\n            if target is None:\n                unmapped[row["label"]] += 1\n                continue\n            scored.append({"relative_path": row["relative_path"], "label": row["label"], "target": target})\n    return scored, unmapped\n\n\ndef crop_of(label: str) -> str:\n    """\'Tomato___Early_blight\' -> \'Tomato\'. PlantDoc labels differ, so both are handled."""\n    name = label.split("___")[0] if "___" in label else label\n    return name.replace("_", " ").replace("(maize)", "").replace(", bell", "").strip().lower()\n\n\ndef bootstrap_ci(correct: np.ndarray, rounds: int, seed: int = 0):\n    """Percentile bootstrap over images, for accuracy."""\n    rng = np.random.default_rng(seed)\n    n = len(correct)\n    if n == 0:\n        return (float("nan"), float("nan"))\n    means = np.empty(rounds)\n    for r in range(rounds):\n        means[r] = correct[rng.integers(0, n, n)].mean()\n    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))\n\n\ndef per_class_report(y_true, y_pred, class_names, num_classes):\n    rows = []\n    for index, name in enumerate(class_names):\n        support = int((y_true == index).sum())\n        if support == 0:\n            continue\n        correct = int(((y_true == index) & (y_pred == index)).sum())\n        predicted = int((y_pred == index).sum())\n        precision = correct / predicted if predicted else 0.0\n        recall = correct / support\n        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0\n        rows.append({"class": name, "support": support, "correct": correct, "recall": recall,\n                     "precision": precision, "f1": f1, "predicted": predicted})\n    return rows\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--arch", required=True)\n    parser.add_argument("--map", type=Path,\n                        default=ROOT / "data" / "manifests" / "plantdoc_to_plantvillage_map.csv")\n    parser.add_argument("--manifest", type=Path,\n                        default=ROOT / "data" / "manifests" / "plantdoc_test.csv")\n    parser.add_argument("--image-root", type=Path, default=ROOT / "data" / "raw" / "PlantDoc" / "test")\n    parser.add_argument("--split", default="test", choices=("test", "all"))\n    parser.add_argument("--bootstrap", type=int, default=2000)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--device", default="cpu")\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n\n    args.out.mkdir(parents=True, exist_ok=True)\n    label_map = load_map(args.map)\n    rows, unmapped = load_rows(args.manifest, args.split, label_map)\n    if not rows:\n        raise SystemExit(f"no scored images: manifest {args.manifest} split {args.split}")\n\n    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)\n    class_names = list(checkpoint["classes"])\n    if len(class_names) != 38:\n        raise SystemExit(f"checkpoint has {len(class_names)} classes, expected 38")\n    image_size = int(checkpoint.get("image_size", 224))\n    index_of = {name: i for i, name in enumerate(class_names)}\n\n    missing = sorted({row["target"] for row in rows if row["target"] not in index_of})\n    if missing:\n        raise SystemExit(f"mapped targets absent from the checkpoint: {missing}")\n\n    model = build_model(args.arch, len(class_names), pretrained=False)\n    state = checkpoint["model_state_dict"]\n    model.load_state_dict({k.replace("module.", "", 1): v for k, v in state.items()})\n    model.eval().to(args.device)\n\n    y_true = np.array([index_of[row["target"]] for row in rows])\n    dataset = PlantDocImages(rows, args.image_root, image_size)\n    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=0)\n\n    predictions = np.empty(len(rows), dtype=np.int64)\n    confidences = np.empty(len(rows), dtype=np.float32)\n    seen = 0\n    with torch.no_grad():\n        for images, indices in loader:\n            logits = model(images.to(args.device))\n            probabilities = logits.softmax(dim=1)\n            confidence, predicted = probabilities.max(dim=1)\n            predictions[indices.numpy()] = predicted.cpu().numpy()\n            confidences[indices.numpy()] = confidence.cpu().numpy()\n            seen += len(indices)\n            print(f"\\r  scored {seen}/{len(rows)}", end="", flush=True)\n    print()\n\n    correct = (predictions == y_true).astype(np.float64)\n    accuracy = float(correct.mean())\n    low, high = bootstrap_ci(correct, args.bootstrap)\n\n    # Classes the model is allowed to answer with: those PlantDoc can actually contain.\n    allowed = {index_of[target] for target in set(label_map.values()) if target in index_of}\n    out_of_mapping = np.array([p not in allowed for p in predictions])\n    out_of_mapping_rate = float(out_of_mapping.mean())\n\n    per_class = per_class_report(y_true, predictions, class_names, len(class_names))\n    macro_f1 = float(np.mean([row["f1"] for row in per_class])) if per_class else float("nan")\n    macro_recall = float(np.mean([row["recall"] for row in per_class])) if per_class else float("nan")\n    weighted_f1 = (float(np.average([row["f1"] for row in per_class],\n                                    weights=[row["support"] for row in per_class]))\n                   if per_class else float("nan"))\n\n    crop_rows = defaultdict(lambda: {"support": 0, "correct": 0})\n    for row, ok in zip(rows, correct):\n        crop = crop_of(row["label"])\n        crop_rows[crop]["support"] += 1\n        crop_rows[crop]["correct"] += int(ok)\n\n    summary = {\n        "checkpoint": str(args.checkpoint),\n        "architecture": args.arch,\n        "dataset": "PlantDoc",\n        "split": args.split,\n        "images_scored": len(rows),\n        "plantdoc_classes_scored": len(per_class),\n        "plantdoc_classes_unmapped": dict(unmapped),\n        "accuracy": accuracy,\n        "accuracy_ci95": [low, high],\n        "ci_width_pp": round((high - low) * 100, 2),\n        "macro_f1": macro_f1,\n        "macro_recall": macro_recall,\n        "weighted_f1": weighted_f1,\n        "mean_confidence": float(confidences.mean()),\n        "out_of_mapping_rate": out_of_mapping_rate,\n        "out_of_mapping_count": int(out_of_mapping.sum()),\n        "mean_over_classes_present_only": True,\n        "class_count_note": "macro averages use only classes with non-zero support in this split",\n    }\n    (args.out / "field_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n\n    with (args.out / "per_class.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(per_class[0]))\n        writer.writeheader()\n        writer.writerows(sorted(per_class, key=lambda r: r["recall"]))\n\n    with (args.out / "per_crop.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["crop", "support", "correct", "recall"])\n        for crop, value in sorted(crop_rows.items()):\n            writer.writerow([crop, value["support"], value["correct"], value["correct"] / value["support"]])\n\n    confusion = np.zeros((len(class_names), len(class_names)), dtype=np.int64)\n    for t, p in zip(y_true, predictions):\n        confusion[t, p] += 1\n    with (args.out / "confusion.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["true\\\\predicted"] + class_names)\n        for index, name in enumerate(class_names):\n            if confusion[index].sum() == 0:\n                continue\n            writer.writerow([name] + confusion[index].tolist())\n\n    with (args.out / "predictions.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["relative_path", "plantdoc_label", "true_class", "predicted_class", "confidence", "correct"])\n        for row, t, p, c in zip(rows, y_true, predictions, confidences):\n            writer.writerow([row["relative_path"], row["label"], class_names[t], class_names[p],\n                             round(float(c), 4), int(t == p)])\n\n    print(f"\\n{args.arch} on PlantDoc ({args.split})")\n    print(f"  images scored      {len(rows):,} across {len(per_class)} classes")\n    print(f"  accuracy           {accuracy:.4f}  95% CI [{low:.4f}, {high:.4f}]  width {summary[\'ci_width_pp\']} pp")\n    print(f"  macro-F1           {macro_f1:.4f}  (classes present only)")\n    print(f"  out-of-mapping     {out_of_mapping_rate:.4f}  ({int(out_of_mapping.sum())} predictions on classes PlantDoc lacks)")\n    print(f"  mean confidence    {float(confidences.mean()):.4f}")\n    if unmapped:\n        print(f"  unmapped images    {sum(unmapped.values())} across {len(unmapped)} labels")\n    print(f"\\nworst classes by recall:")\n    for row in sorted(per_class, key=lambda r: r["recall"])[:5]:\n        print(f"  {row[\'class\']:52s} recall {row[\'recall\']:.2f}  support {row[\'support\']}")\n    print(f"\\nwrote {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/paired_bootstrap.py': '#!/usr/bin/env python3\n"""Paired bootstrap on the PlantVillage to PlantDoc drop (roadmap S19 gate).\n\nTwo models are compared on the same 236 field images, so the comparison is\npaired: for each image both models either succeed or fail, and what matters is the\ndifference in the drop from the laboratory benchmark.\n\nTwo quantities are reported and they answer different questions.\n\n**Field accuracy difference.** How much better is the treatment in the field,\nsigned. Bootstrap over images, resampling the *same* images for both models so the\npairing is respected.\n\n**Drop difference.** How much less does the treatment lose going from lab to\nfield. This is the H3 quantity: if augmentation helps, the treatment\'s drop should\nbe smaller than the baseline\'s.\n\nReporting only the first would let a treatment look good while actually being worse\nin the lab. Reporting both is the honest pairing.\n\n    python scripts/paired_bootstrap.py \\\n        --baseline reports/field_gap/b0_plantdoc_test/predictions.csv \\\n        --treatment reports/field_gap/composite_b0_plantdoc_test/predictions.csv \\\n        --baseline-lab 0.99280 --treatment-lab 0.99 \\\n        --out reports/field_gap/paired_b0.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef load(path: Path) -> dict[str, tuple[int, float]]:\n    """relative_path -> (correct, confidence). Keyed on path so pairing is exact."""\n    rows = {}\n    with path.open(encoding="utf-8") as handle:\n        for row in csv.DictReader(handle):\n            rows[row["relative_path"]] = (int(row["correct"] == "1"), float(row["confidence"]))\n    return rows\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--baseline", type=Path, required=True)\n    parser.add_argument("--treatment", type=Path, required=True)\n    parser.add_argument("--baseline-lab", type=float, required=True,\n                        help="baseline PlantVillage accuracy")\n    parser.add_argument("--treatment-lab", type=float, required=True,\n                        help="treatment PlantVillage accuracy")\n    parser.add_argument("--rounds", type=int, default=5000)\n    parser.add_argument("--seed", type=int, default=0)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n\n    base, treat = load(args.baseline), load(args.treatment)\n    shared = sorted(set(base) & set(treat))\n    if len(shared) != len(base) or len(shared) != len(treat):\n        raise SystemExit(\n            f"the two prediction files cover different images: baseline {len(base)}, "\n            f"treatment {len(treat)}, shared {len(shared)}. A paired test needs identical images."\n        )\n\n    b_ok = np.array([base[p][0] for p in shared], dtype=np.float64)\n    t_ok = np.array([treat[p][0] for p in shared], dtype=np.float64)\n    n = len(shared)\n\n    base_acc, treat_acc = b_ok.mean(), t_ok.mean()\n    base_drop = args.baseline_lab - base_acc\n    treat_drop = args.treatment_lab - treat_acc\n\n    rng = np.random.default_rng(args.seed)\n    deltas = np.empty(args.rounds)\n    drop_deltas = np.empty(args.rounds)\n    for r in range(args.rounds):\n        idx = rng.integers(0, n, n)          # same resample for both models\n        deltas[r] = t_ok[idx].mean() - b_ok[idx].mean()\n        drop_deltas[r] = ((args.baseline_lab - b_ok[idx].mean())\n                          - (args.treatment_lab - t_ok[idx].mean()))\n\n    def ci(sample):\n        return [float(np.percentile(sample, 2.5)), float(np.percentile(sample, 97.5))]\n\n    both_wrong = int(((b_ok == 0) & (t_ok == 0)).sum())\n    treatment_fixes = int(((b_ok == 0) & (t_ok == 1)).sum())\n    treatment_breaks = int(((b_ok == 1) & (t_ok == 0)).sum())\n\n    summary = {\n        "images": n,\n        "baseline_lab_accuracy": args.baseline_lab,\n        "treatment_lab_accuracy": args.treatment_lab,\n        "baseline_field_accuracy": round(float(base_acc), 6),\n        "treatment_field_accuracy": round(float(treat_acc), 6),\n        "field_delta_pp": round(float((treat_acc - base_acc) * 100), 2),\n        "field_delta_ci95_pp": [round(v * 100, 2) for v in ci(deltas)],\n        "baseline_drop_pp": round(float(base_drop * 100), 2),\n        "treatment_drop_pp": round(float(treat_drop * 100), 2),\n        "drop_improvement_pp": round(float((base_drop - treat_drop) * 100), 2),\n        "drop_improvement_ci95_pp": [round(v * 100, 2) for v in ci(drop_deltas)],\n        "rounds": args.rounds,\n        "paired_movements": {\n            "both_wrong": both_wrong,\n            "treatment_fixes": treatment_fixes,\n            "treatment_breaks": treatment_breaks,\n        },\n        "verdict": None,\n    }\n\n    low, high = summary["drop_improvement_ci95_pp"]\n    if low > 0:\n        summary["verdict"] = "H3 supported: the drop is smaller for the treatment, CI excludes zero"\n    elif high < 0:\n        summary["verdict"] = "H3 contradicted: the treatment\'s drop is larger, CI excludes zero"\n    else:\n        summary["verdict"] = "H3 not supported: the CI on the drop improvement includes zero"\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    args.out.write_text(json.dumps(summary, indent=2), encoding="utf-8")\n\n    print(f"images                    {n}")\n    print(f"baseline field accuracy   {base_acc:.4f}  (lab {args.baseline_lab:.4f}, drop {base_drop*100:.1f} pp)")\n    print(f"treatment field accuracy  {treat_acc:.4f}  (lab {args.treatment_lab:.4f}, drop {treat_drop*100:.1f} pp)")\n    print(f"field delta               {summary[\'field_delta_pp\']:+.2f} pp  "\n          f"CI {summary[\'field_delta_ci95_pp\']}")\n    print(f"drop improvement          {summary[\'drop_improvement_pp\']:+.2f} pp  "\n          f"CI {summary[\'drop_improvement_ci95_pp\']}")\n    print(f"paired movements          fixes {treatment_fixes}, breaks {treatment_breaks}, "\n          f"both wrong {both_wrong}")\n    print(f"\\n{summary[\'verdict\']}")\n\n\nif __name__ == "__main__":\n    main()\n',
}

for _path, _content in FILES.items():
    _target = Path(_path)
    _target.parent.mkdir(parents=True, exist_ok=True)
    _target.write_text(_content, encoding='utf-8')
print(f'wrote {len(FILES)} files')

import py_compile
for _file in (SCRIPT, WORKER, MASK_SCRIPT, COMPOSITE_SCRIPT, DISJOINT_SCRIPT, EVAL_SCRIPT, BOOTSTRAP_SCRIPT):
    py_compile.compile(str(_file), doraise=True)
print('all scripts compile')

PROTOCOL = json.loads(PROTOCOL_PATH.read_text())
assert PROTOCOL['version'] == 2 and PROTOCOL['shuffle_mode'] == 'random'
b0 = PROTOCOL['architectures']['efficientnet_b0']
assert b0['lr'] == 1e-4 and b0['local_batch_size'] == 16, b0
print('protocol v2 unchanged:', b0)

## 5. Gate, then six runs

The gate trains baseline seed 42 for two epochs and requires 0.95 validation accuracy. It writes into its own directory, so it cannot interfere with the six real runs.

If the gate passes, both arms train three seeds each. A failure in one arm does not stop the other.

**Expected output:** the gate history, then a line per completed run. Each run is about 25 minutes.

In [ ]:
cache_args = [
    "--train-labels", str(TRAIN_Y_PATH),
    "--val-images", str(VAL_X_PATH), "--val-labels", str(VAL_Y_PATH),
    "--classes-json", str(CLASSES_PATH), "--protocol", str(PROTOCOL_PATH),
]


def worker(arch, seeds, epochs, output_root, train_images):
    return [sys.executable, str(WORKER), "--arch", arch, "--mode", "ddp",
            "--seeds", *[str(s) for s in seeds], "--epochs", str(epochs),
            "--train-images", str(train_images), "--output-root", str(output_root), *cache_args]


ARMS = {
    "baseline":  {"train_images": TRAIN_X_PATH,     "root": OUTPUT_ROOT / "baseline"},
    "composite": {"train_images": COMPOSITE_X,      "root": OUTPUT_ROOT / "composite"},
}

# The gate trains into its own directory. If it wrote into the baseline arm, it
# would leave a validation_metrics.json behind, the worker skips seeds that
# already have one, and baseline seed 42 would never train the full 12 epochs.
# The comparison would then be 2-epoch baseline against 12-epoch treatment.
GATE_ROOT = OUTPUT_ROOT / "_gate"
if GATE_ROOT.exists():
    shutil.rmtree(GATE_ROOT)

print("GATE: baseline seed 42, 2 epochs, must reach "
      f"{GATE_ACCURACY} before six runs are committed")
gate = subprocess.run(worker("efficientnet_b0", [SEEDS[0]], 2, GATE_ROOT,
                             ARMS["baseline"]["train_images"]),
                      env={**os.environ, "PYTHONUNBUFFERED": "1"})
gate_history = pd.read_csv(GATE_ROOT / "efficientnet_b0" / f"seed{SEEDS[0]}"
                           / "training_history.csv")
accuracy = float(gate_history["val_accuracy"].iloc[-1])
print(gate_history[["epoch", "train_loss", "val_accuracy", "lr"]].to_string(index=False))
print(f"gate accuracy after 2 epochs: {accuracy:.4f}")

GATE_PASSED = gate.returncode == 0 and accuracy >= GATE_ACCURACY
if not GATE_PASSED:
    print("GATE FAILED - no long runs tonight. The gate evidence is still in the zip below.")

if GATE_PASSED:
    for arm, spec in ARMS.items():
        print(f"\n===== {arm} : {len(SEEDS)} seeds =====", flush=True)
        result = subprocess.run(worker("efficientnet_b0", SEEDS, EPOCHS, spec["root"],
                                       spec["train_images"]),
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        if result.returncode != 0:
            print(f"  {arm} arm returned {result.returncode}, continuing so partial results survive")
    print(f"\ntraining finished in {(time.time() - STARTED_AT)/3600:.2f} h")

## 6. Field evaluation

Scores every trained checkpoint on PlantDoc: 236 images, 27 mapped classes, never seen during training.

**Expected output:** one line per checkpoint with its field accuracy. Baseline should land near 0.21 to 0.25, consistent with the 0.2085 five-seed mean already on record.

In [ ]:
if not GATE_PASSED:
    print("skipped: gate did not pass")
else:
    EVALS = {}
    for arm, spec in ARMS.items():
        for seed in SEEDS:
            checkpoint = spec["root"] / "efficientnet_b0" / f"seed{seed}" / "efficientnet_b0_best.pt"
            out_dir = OUTPUT_ROOT / "field" / arm / f"seed{seed}"
            if not checkpoint.is_file():
                print(f"  missing checkpoint: {checkpoint}")
                continue
            if (out_dir / "predictions.csv").is_file():
                print(f"  {arm} seed {seed}: cached")
            else:
                subprocess.run([sys.executable, str(EVAL_SCRIPT), "--checkpoint", str(checkpoint),
                                "--arch", "efficientnet_b0", "--split", "test", "--device", "cuda",
                                "--bootstrap", "1000", "--out", str(out_dir)], check=False)
            metrics_path = out_dir / "field_metrics.json"
            if metrics_path.is_file():
                m = json.loads(metrics_path.read_text())
                EVALS[(arm, seed)] = {"field_accuracy": m["accuracy"],
                                      "predictions": out_dir / "predictions.csv"}
                print(f"  {arm:10s} seed {seed}  field accuracy {m['accuracy']:.4f}")

## 7. Paired comparison

For each seed, compares the two arms on the same field images with a paired bootstrap, 5,000 rounds. Reports the difference in field accuracy **and** the difference in the drop from PlantVillage, because a model can look better in the field while being worse in the lab.

**Expected output:** a table, then one verdict per seed. The verdict is the H3 answer: supported if the confidence interval on the drop improvement excludes zero.

In [ ]:
if not GATE_PASSED or not EVALS:
    print("nothing to compare yet")
else:
    protocol = json.loads(PROTOCOL_PATH.read_text())
    results = []
    for seed in SEEDS:
        if ("baseline", seed) not in EVALS or ("composite", seed) not in EVALS:
            continue
        base_metrics = json.loads((OUTPUT_ROOT / "field" / "baseline" / f"seed{seed}"
                                   / "field_metrics.json").read_text())
        treat_metrics = json.loads((OUTPUT_ROOT / "field" / "composite" / f"seed{seed}"
                                    / "field_metrics.json").read_text())
        base_lab = json.loads((ARMS["baseline"]["root"] / "efficientnet_b0" / f"seed{seed}"
                               / "validation_metrics.json").read_text())["val_accuracy"]
        treat_lab = json.loads((ARMS["composite"]["root"] / "efficientnet_b0" / f"seed{seed}"
                                / "validation_metrics.json").read_text())["val_accuracy"]

        out_json = OUTPUT_ROOT / "paired" / f"seed{seed}.json"
        subprocess.run([sys.executable, str(BOOTSTRAP_SCRIPT),
                        "--baseline", str(EVALS[("baseline", seed)]["predictions"]),
                        "--treatment", str(EVALS[("composite", seed)]["predictions"]),
                        "--baseline-lab", str(base_lab), "--treatment-lab", str(treat_lab),
                        "--out", str(out_json)], check=False)
        if out_json.is_file():
            row = json.loads(out_json.read_text())
            row["seed"] = seed
            row["baseline_lab_accuracy"] = base_lab
            row["treatment_lab_accuracy"] = treat_lab
            results.append(row)

    if not results:
        print("no paired seed completed")
    else:
        table = pd.DataFrame([{
            "seed": r["seed"],
            "lab baseline": round(r["baseline_lab_accuracy"], 4),
            "lab composite": round(r["treatment_lab_accuracy"], 4),
            "field baseline": round(r["baseline_field_accuracy"], 4),
            "field composite": round(r["treatment_field_accuracy"], 4),
            "drop baseline pp": round(r["baseline_drop_pp"], 1),
            "drop composite pp": round(r["treatment_drop_pp"], 1),
            "drop improvement pp": r["drop_improvement_pp"],
            "CI": r["drop_improvement_ci95_pp"],
        } for r in results])
        display(table)
        (OUTPUT_ROOT / "paired_summary.csv").write_text(table.to_csv(index=False))

        supported = sum(1 for r in results if r["verdict"].startswith("H3 supported"))
        print(f"\nseeds where the composite arm reduced the drop: {supported}/{len(results)}")
        for r in results:
            print(f"  seed {r['seed']}: {r['verdict']}")

        label = "s19" if len(SEEDS) > 1 else "s19_smoke"
        zip_path = shutil.make_archive(f"/kaggle/working/{label}_results", "zip",
                                       root_dir=OUTPUT_ROOT)
        print(f"\nBundle: {zip_path} ({Path(zip_path).stat().st_size/1e6:.0f} MB)")

## 8. Bundle

Runs whatever happened above. Writes an artifact manifest and zips the run directory, so a failed gate or a crashed arm still leaves something to diagnose from.

In [ ]:
# Always ship something. If the gate failed or a run crashed, this is the evidence.
manifest = {}
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest[str(path.relative_to(OUTPUT_ROOT))] = round(path.stat().st_size / 1e6, 2)
(OUTPUT_ROOT / "artifact_manifest.json").write_text(json.dumps({
    "files": manifest,
    "total_files": len(manifest),
    "session_hours": round((time.time() - STARTED_AT) / 3600, 2),
    "gate_passed": bool(GATE_PASSED),
    "seeds": SEEDS,
    "epochs": EPOCHS,
}, indent=2), encoding="utf-8")

print(f"{len(manifest)} artifacts in the run directory")
for name in list(manifest)[:15]:
    print(f"  {name}")
if len(manifest) > 15:
    print(f"  ... and {len(manifest) - 15} more")

bundle = shutil.make_archive("/kaggle/working/s19_results", "zip", root_dir=OUTPUT_ROOT)
print("Bundle: " + bundle + f"  ({Path(bundle).stat().st_size/1e6:.0f} MB)")
print("Download it from the Output panel on the right.")
if not GATE_PASSED:
    print("Note: the gate did not pass, so this bundle contains gate evidence, not results.")

## Reading the result

A positive result means the composite arm's drop is smaller and the confidence
interval on the improvement excludes zero. Report it as "recovered X points of the
regression, 95% CI [a, b]", never as "solved".

A null result is equally publishable and pairs naturally with the calibration
finding: if randomising the background does not help, the gap is about leaf
appearance across domains rather than the scene around the leaf.

Either way, the result goes into `results/registry.json` with this run's artifact
before it appears in any document. And the two method limits at the top of this
notebook travel with the number.

## If something goes wrong

**"Background corpus not attached"** in cell 1. Add the DeepWeeds dataset from the
right-hand panel, using the slug `imsparsh/deepweeds`, then Run All again.

**"DISJOINTNESS FAILED"** in cell 2. A background image is byte-identical to a
PlantDoc or PlantVillage image. Do not work around this. Check
`/kaggle/temp/derived/disjointness.json` for which file collided, and either remove
that file from the corpus or use a different corpus. Training on it would put the
evaluation set into the training data.

**"GATE FAILED"** in cell 5. The baseline arm could not reach 0.95 validation
accuracy in two epochs, which means the recipe is not reproducing the reference
behaviour (the original reached 0.9836 in epoch 1). Nothing long will run. Send me
the gate history printed above the error and I will diagnose it. Do not raise the
gate threshold and do not re-run hoping for a different outcome.

**A run shows `[FAIL]` or `[TIMEOUT]`** in cell 5. That seed is skipped and the
loop continues. The other seeds still run. Note which one failed and send the
bundle; one missing seed is not fatal.

**Field evaluation prints `missing checkpoint`** in cell 6. That arm's training did
not finish. The remaining pairs still compare.

**"no paired seed completed"** in cell 7. Both arms are needed for a pair. Check
that cell 5 printed results for both `baseline` and `composite`.

**Runtime is longer than expected.** Three seeds per arm is six runs. Set
`SEEDS = [42]` at the top of cell 1 and re-run; finished work is not repeated.

## What to send back

When the run finishes, the last cell prints the bundle path and size. Download
`/kaggle/working/s19_results.zip` from the Output panel on the right and send it
over.

The bundle contains, for each arm and seed, the checkpoint, the training history,
the validation metrics and the field predictions, plus one `paired/seed<N>.json`
per comparison with the confidence intervals.

With that in hand I can register the H3 verdict, update every document that
currently says "blocked", and fold the result into the results section. Until then
H3 stays marked as not tested, which is what the submission already says.